In [ ]:
!pip show transformers

In [ ]:
!pip uninstall -y -q transformers
!pip uninstall -y -q transformers
!rm -rf /usr/local/lib/python3.13/dist-packages/transformers /usr/local/lib/python3.13/dist-packages/transformers-*.dist-info
!pip install -q --no-cache-dir "transformers==4.57.6" "huggingface_hub>=0.34,<1.0"

In [ ]:
import transformers, huggingface_hub
print(transformers.__version__, huggingface_hub.__version__)
from transformers import SegformerConfig, SegformerForSemanticSegmentation
print("SegFormer import OK")

In [ ]:
import transformers, huggingface_hub
print("transformers   :", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 1 — DEPENDENCY INSTALLATION
# SegFormer (MiT-B2) + DINOv3 / BYOL / MAE self-supervised pretraining + baseline — 2 epochs
# Same notebook structure as the YOLO26s notebook.
# ============================================================
import sys
import subprocess

packages = [
    "transformers>=4.56.0,<5",
    "huggingface_hub>=0.34,<1.0",
    "Pillow>=10.0,<13.0",
    "lime==0.2.0.1",
    "opencv-python-headless",
    "scikit-image",
    "scikit-learn",
    "pyyaml",
    "tqdm",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("Required project packages installed.")


In [ ]:
import sys
import subprocess

# Upgrade transformers to a stable version compatible with Python 3.13
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "transformers>=4.48.3", "huggingface_hub>=0.28.0"],
    check=True,
)

import transformers
print("Upgraded transformers version:", transformers.__version__)

In [ ]:
# ============================================================
# CELL 2 — SETUP: IMPORTS, SEED, DEVICE, PATHS, CONSTANTS, PLOT STYLE
# Purpose: Step 1 environment initialisation (seed = 42, device, versions).
# The ONLY place to change the epoch count is EPOCHS below.
# ============================================================
import os, gc, json, math, random, shutil, zipfile, time, platform
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import yaml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import SegformerConfig, SegformerForSemanticSegmentation

# ---- run configuration ----
EPOCHS = 100                                 # <<< the ONLY place to change the epoch count
SSL_EPOCHS = EPOCHS                         # Stage A: self-supervised pretraining epochs
FT_EPOCHS = EPOCHS                          # Stage B: supervised fine-tuning epochs
SEED = 42                                   # Step 1: seed = 42
NC = 6                                      # six disease / health classes
NUM_CLASSES = NC + 1                        # + background (class 0)
IMG_SIZE = 512                              # SegFormer 512 crop / input size (Step 2)
FT_BATCH = 8                                # SegFormer pilot value
WORKERS = 2
SSL_SIZE = 224                               # SSL crop size (same as the YOLO26s notebook)
SSL_BATCH = 32                              # BYOL batch (two views, online + target)
TEACHER_BATCH = 64                          # DINOv3 / MAE teacher batch
LR_ENCODER = 6e-5                           # AdamW, encoder (pilot value)
LR_HEAD = 6e-4                              # AdamW, decode head (pilot value)
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1                       # share of iterations used for linear warm-up
BASELINE_INIT = "random"                    # "random" = same as the YOLO26s baseline (random init)
                                            # "imagenet" = ImageNet-pretrained MiT-B2 (needs internet)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
EPOCH_TICKS = (list(range(1, FT_EPOCHS + 1)) if FT_EPOCHS <= 20
               else [1] + list(range(10, FT_EPOCHS + 1, 10)))
plt.rcParams["lines.markersize"] = 6 if FT_EPOCHS <= 20 else 3

# ---- paths ----
INPUT = Path(os.environ.get("PAPAYA_INPUT", "/kaggle/input"))
W = Path(os.environ.get("PAPAYA_WORK", "/kaggle/working"))
DS = W / "segformer_ds"                     # images + masks copied from the annotated dataset
OUT = W / "segformer"                       # /kaggle/working/<backbone>/<method>/
EXPS = ["baseline", "dino", "byol", "mae"]  # the four runs (same order as the YOLO26s notebook)
EXP_ROOTS = {m: OUT / m for m in EXPS}      # per-run folder: ssl_weights.pt, best.pt, history.csv, ...
CKPT = OUT / "checkpoints"                  # shared SSL checkpoints
LOGS = OUT / "logs"
EV = OUT / "eval"
FIG = EV / "figures"
for _p in [DS, CKPT, LOGS, EV, FIG, *EXP_ROOTS.values()]:
    _p.mkdir(parents=True, exist_ok=True)

DISPLAY_NAMES = {"baseline": "Baseline", "dino": "DINOv3", "byol": "BYOL", "mae": "MAE"}
MODEL_LABELS_1LINE = {m: f"SegFormer + {DISPLAY_NAMES[m]}" if m != "baseline" else "SegFormer baseline"
                      for m in EXPS}
MODEL_COLORS = {"baseline": "#555555", "dino": "#0072B2", "byol": "#D55E00", "mae": "#009E73"}


def seed_all(seed=None):
    seed = SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


seed_all()

# ---- one colour palette, reused in every plot ----
MASK_COLORS = np.array([
    [0, 0, 0],          # 0 background
    [60, 180, 75],      # 1
    [230, 25, 75],      # 2
    [255, 225, 25],     # 3
    [0, 130, 200],      # 4
    [245, 130, 48],     # 5
    [145, 30, 180],     # 6
], dtype=np.uint8)
CLASS_COLORS_7 = ["#222222", "#009E73", "#D55E00", "#B59F00",
                  "#0072B2", "#E69F00", "#CC79A7"]      # background + 6 classes (Okabe-Ito)


def color_mask(mask):
    return MASK_COLORS[np.asarray(mask).astype(np.uint8)]


mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 14,
    "axes.titlesize": 17, "axes.titleweight": "bold",
    "axes.labelsize": 15, "axes.labelweight": "bold",
    "xtick.labelsize": 13, "ytick.labelsize": 13,
    "legend.fontsize": 13, "legend.frameon": True,
    "figure.titlesize": 20, "figure.titleweight": "bold",
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
})

# ---- Step 1: log library and GPU versions ----
import sklearn, transformers
print("python        :", platform.python_version())
print("torch         :", torch.__version__)
print("transformers  :", transformers.__version__)
print("scikit-learn  :", sklearn.__version__)
print("seed          :", SEED)
print("device        :", DEVICE)
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
print("SSL_EPOCHS    :", SSL_EPOCHS, "| FT_EPOCHS:", FT_EPOCHS, "| baseline start weights:", BASELINE_INIT)
print("Output folder  :", OUT)


# ---- session time budget: Kaggle kills a GPU session at 12 h, so stop cleanly before that ----
RUN_START = time.time()
TIME_BUDGET_H = float(os.environ.get("PAPAYA_TIME_BUDGET_H", "11.0"))   # hours this session may train
STOP_TRAINING = False


def time_up(last_epoch_s):
    """True if another epoch (+15 min safety for saving) would not fit in the time budget."""
    global STOP_TRAINING
    if time.time() - RUN_START + 1.2 * last_epoch_s + 15 * 60 > TIME_BUDGET_H * 3600:
        STOP_TRAINING = True
    return STOP_TRAINING


def halt_notebook(where):
    """Skip every remaining cell WITHOUT raising an error, so Kaggle saves /kaggle/working normally."""
    ip = get_ipython()
    if not any(getattr(f, "_papaya_halt", False) for f in ip.input_transformers_post):
        def _skip(lines):
            return ['print("Skipped: time budget reached. Attach this version\'s output and run again.")\n']
        _skip._papaya_halt = True
        ip.input_transformers_post.append(_skip)
    print("=" * 70)
    print(f"TIME BUDGET REACHED during {where} after {(time.time() - RUN_START) / 3600:.2f} h.")
    print("Progress is saved under", OUT, "- all remaining cells will be skipped.")
    print("Next session: Add Input -> this notebook's latest output, then Save & Run All again.")
    print("=" * 70)


print("Time budget   :", TIME_BUDGET_H, "h")


In [ ]:
# ============================================================
# CELL 2b — LOCATE THE TWO INPUTS + RESTORE PROGRESS FROM A PREVIOUS SESSION
# Modified for Google Colab to read directly from mounted Google Drive.
# Added automatic installation of patool to avoid ModuleNotFoundError.
# ============================================================
import sys
import subprocess

try:
    import patoolib
except ModuleNotFoundError:
    print("Installing patool library...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "patool"], check=True)
    import patoolib

import os
import zipfile
import shutil
from pathlib import Path
import torch
import pandas as pd
from IPython.display import display

# Define paths in Google Drive
DRIVE_PROGRESS = Path("/content/drive/MyDrive/Segformer/Segformer.zip")
DRIVE_ANNOTATED = Path("/content/drive/MyDrive/Segformer/new_annotated.zip")

# Colab working directories
colab_working = Path("/content/working")
colab_working.mkdir(parents=True, exist_ok=True)

PROGRESS_EXTRACT = colab_working / "progress_extract"
ANNOTATED_EXTRACT = colab_working / "dataset_extract"

# Extract Progress .rar if not already done
if DRIVE_PROGRESS.exists():
    if not PROGRESS_EXTRACT.exists():
        PROGRESS_EXTRACT.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {DRIVE_PROGRESS}...")
        try:
            patoolib.extract_archive(str(DRIVE_PROGRESS), outdir=str(PROGRESS_EXTRACT))
            print("Progress archive extracted successfully.")
        except Exception as e:
            print(f"Error extracting rar archive: {e}.")
else:
    print(f"WARNING: Progress archive not found at {DRIVE_PROGRESS}")

# Extract Annotated Dataset .zip if not already done
if DRIVE_ANNOTATED.exists():
    if not ANNOTATED_EXTRACT.exists():
        ANNOTATED_EXTRACT.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {DRIVE_ANNOTATED}...")
        with zipfile.ZipFile(DRIVE_ANNOTATED, 'r') as zip_ref:
            zip_ref.extractall(ANNOTATED_EXTRACT)
        print("Annotated dataset extracted successfully.")
else:
    raise FileNotFoundError(f"Annotated dataset zip not found at {DRIVE_ANNOTATED}")

# Locate progress folder inside extraction
def _find_previous_outputs(root):
    found = []
    for dirpath, dirnames, _ in os.walk(root, followlinks=True):
        d = Path(dirpath)
        if d.name.lower() == "segformer" and (
                "checkpoints" in dirnames or any(m in dirnames for m in EXPS)):
            found.append(d)
            dirnames[:] = []
    return found

previous = _find_previous_outputs(PROGRESS_EXTRACT)
if previous:
    prev = max(previous, key=lambda p: sum(1 for f in p.rglob("*") if f.is_file()))
    print("Restoring progress from:", prev)
    copied = 0
    for f in prev.rglob("*"):
        if f.is_file():
            dest = OUT / f.relative_to(prev)
            if not dest.exists():
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dest)
                copied += 1
    print("Files restored:", copied)
else:
    print("No previous progress folder found inside extraction.")

# Rebuild missing SSL checkpoints
for m, final_name in {"dino": "dino_segformer_backbone_final.pt", "byol": "byol_segformer_backbone_final.pt",
                      "mae": "mae_segformer_backbone_final.pt"}.items():
    src_w, dst_f = EXP_ROOTS[m] / "ssl_weights.pt", CKPT / final_name
    if src_w.exists() and not dst_f.exists():
        ck = torch.load(src_w, map_location="cpu", weights_only=False)
        if ck.get("epochs") == SSL_EPOCHS:
            shutil.copy2(src_w, dst_f)
            print(f"Rebuilt {final_name} from {m}/ssl_weights.pt")

# Progress report
rows = []
for name, final in [("SSL dino", "dino_segformer_backbone_final.pt"), ("SSL byol", "byol_segformer_backbone_final.pt"),
                    ("SSL mae", "mae_segformer_backbone_final.pt")]:
    rows.append({"stage": name, "status": "done" if (CKPT / final).exists() else "NOT FOUND - will retrain"})
for m in EXPS:
    root = EXP_ROOTS[m]
    if (root / "DONE.json").exists():
        st = "done"
    elif (root / "history.csv").exists():
        st = f"epoch {int(pd.read_csv(root / 'history.csv')['epoch'].max())}/{FT_EPOCHS}"
    else:
        st = "not started"
    rows.append({"stage": f"fine-tune {m}", "status": st})
display(pd.DataFrame(rows).set_index("stage"))

# Set final inputs pointing to the newly extracted annotated folder
INPUT = ANNOTATED_EXTRACT

In [ ]:
# ============================================================
# CELL 3 — LOCATE AND EXTRACT THE ANNOTATED DATASET
# Purpose: Find papaya_annotated or annotated.zip in Kaggle.
# ============================================================

def find_dataset_root(search_root):
    for file in search_root.rglob("class_names.json"):
        root = file.parent
        if all((root / name).exists() for name in ["images", "masks", "manifests"]):
            return root
    return None


ROOT = find_dataset_root(INPUT)

if ROOT is None:
    archives = list(INPUT.rglob("annotated.zip"))

    if archives:
        extraction_dir = W / "dataset_extract"
        extraction_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.namelist():
                destination = (extraction_dir / item).resolve()
                assert destination.is_relative_to(extraction_dir.resolve())
            archive.extractall(extraction_dir)

        ROOT = find_dataset_root(extraction_dir)

# An already-prepared SegFormer dataset can be reused.
PREPARED = (
    (DS / "split_table.csv").exists()
    and (DS / "class_map.json").exists()
)

assert ROOT is not None or PREPARED, "Attach annotated.zip or the prepared dataset."

print("Original dataset:", ROOT)
print("Prepared SegFormer dataset available:", PREPARED)


In [ ]:
# ============================================================
# CELL 4 — DATASET VALIDATION (Step 1 integrity checks)
# Purpose: Verify split sizes, leakage, image/mask pairs, shapes and mask values 0..6.
# ============================================================
SPLITS = ["train", "val", "test"]

if ROOT is not None:
    man = pd.concat(
        [pd.read_csv(ROOT / "manifests" / f"{s}.csv") for s in SPLITS],
        ignore_index=True,
    )
    class_ids = json.loads((ROOT / "class_names.json").read_text())

    MASK_NAMES = {0: "background"}
    MASK_NAMES.update({int(v): k for k, v in class_ids.items() if int(v) > 0})
else:
    man = pd.read_csv(DS / "split_table.csv")
    class_map = json.loads((DS / "class_map.json").read_text())
    MASK_NAMES = {int(k): v for k, v in class_map["mask_value_to_name"].items()}

man["uid"] = man["uid"].astype(str)
man["class_name"] = man["label"].astype(int).map(MASK_NAMES)
CLASS_NAMES = [MASK_NAMES[i] for i in range(1, NUM_CLASSES)]          # six disease / health classes

print("Dataset size:", len(man))
print("\nSplit counts:")
print(man["split"].value_counts())

assert man["uid"].is_unique
assert len(man) == 1358
assert set(man["split"]) == set(SPLITS)
assert sorted(MASK_NAMES) == list(range(NUM_CLASSES)), "Expected mask values 0..6"
assert man["class_name"].notna().all()

for key in ["group", "md5", "source_image"]:
    if key in man.columns:
        leakage = man.groupby(key)["split"].nunique().gt(1).sum()
        print(f"{key} cross-split leakage:", leakage)
        assert leakage == 0

# Validate image / mask pairs where the original dataset exists.
if ROOT is not None:
    for row in tqdm(man.itertuples(index=False), total=len(man), desc="Checking image/mask pairs"):
        for col in ["image", "mask"]:
            assert (ROOT / getattr(row, col)).exists(), (col, row.uid)

        mask = np.asarray(Image.open(ROOT / row.mask))
        if mask.ndim == 3:
            mask = mask[..., 0]

        expected = int(row.label)
        values = set(np.unique(mask).tolist())
        assert values.issubset({0, expected}), row.uid                 # also guarantees 0..6
        assert min(values) >= 0 and max(values) <= NC, row.uid

        width, height = Image.open(ROOT / row.image).size              # header only
        assert mask.shape == (height, width), f"shape mismatch: {row.uid}"

print("\nDataset validation completed.")


In [ ]:
# ============================================================
# CELL 5 — BUILD THE SEGFORMER DATASET FOLDERS
# Purpose: Copy images and class-index masks into one folder per split.
# (SegFormer needs no polygon labels, so only images/ and masks/ are made.)
# ============================================================
EXPECTED_COUNTS = {"train": 944, "val": 207, "test": 207}


def dataset_complete():
    for split, expected in EXPECTED_COUNTS.items():
        for sub in ["images", "masks"]:
            folder = DS / sub / split
            if not folder.exists() or len(list(folder.iterdir())) != expected:
                return False
    return True


if dataset_complete():
    print("Existing SegFormer dataset found. Skipping file copies.")
else:
    assert ROOT is not None, "Original annotated dataset required for copying."

    for split in SPLITS:
        for sub in ["images", "masks"]:
            (DS / sub / split).mkdir(parents=True, exist_ok=True)

    for row in tqdm(man.itertuples(index=False), total=len(man), desc="Preparing files"):
        uid = str(row.uid)
        extension = Path(row.image).suffix.lower()
        shutil.copy2(ROOT / row.image, DS / "images" / row.split / f"{uid}{extension}")
        shutil.copy2(ROOT / row.mask, DS / "masks" / row.split / f"{uid}.png")

assert dataset_complete()

(DS / "class_map.json").write_text(json.dumps({"mask_value_to_name": MASK_NAMES}, indent=2))
man.to_csv(DS / "split_table.csv", index=False)

for split in SPLITS:
    print(split, {sub: len(list((DS / sub / split).iterdir())) for sub in ["images", "masks"]})
# Self-supervised stage may read ONLY these files (training split; no labels, no val, no test).
SSL_FILES = sorted(str(p) for p in (DS / "images" / "train").iterdir() if p.is_file())
(DS / "ssl_train_images.txt").write_text("\n".join(SSL_FILES))
assert len(SSL_FILES) == EXPECTED_COUNTS["train"]
assert all("/images/train/" in p.replace("\\", "/") for p in SSL_FILES)
print("SSL training images (train split only):", len(SSL_FILES))
print("Dataset preparation complete.")


In [ ]:
# ============================================================
# CELL 6 — DATASET SUMMARY + ONE IMAGE / MASK / OVERLAY PER CLASS
# Purpose: Print images per split and class, the image size range,
#          and show one example per class (image, mask, overlay).
# ============================================================
IMAGE_INDEX = {
    split: {p.stem: p for p in sorted((DS / "images" / split).iterdir()) if p.is_file()}
    for split in SPLITS
}


def load_rgb(path):
    return np.asarray(Image.open(path).convert("RGB"))


def load_gt(split, uid):
    gt = np.asarray(Image.open(DS / "masks" / split / f"{uid}.png"))
    return gt[..., 0] if gt.ndim == 3 else gt


print("Images per split and class:")
display(pd.crosstab(man["class_name"], man["split"])[SPLITS])

sizes = np.array([Image.open(p).size for s in SPLITS for p in IMAGE_INDEX[s].values()])
print(f"Image width  range: {sizes[:, 0].min()} - {sizes[:, 0].max()}")
print(f"Image height range: {sizes[:, 1].min()} - {sizes[:, 1].max()}")

fig, axes = plt.subplots(NC, 3, figsize=(11, 3.4 * NC), squeeze=False)
for row_i, class_name in enumerate(CLASS_NAMES):
    sample = man[(man["split"] == "train") & (man["class_name"] == class_name)].iloc[0]
    rgb = load_rgb(IMAGE_INDEX["train"][sample["uid"]])
    gt = load_gt("train", sample["uid"])
    overlay = np.where((gt > 0)[..., None],
                       (rgb * 0.5 + color_mask(gt) * 0.5).astype(np.uint8), rgb)
    for col_i, (panel, title) in enumerate(zip(
            [rgb, color_mask(gt), overlay], ["Image", "Mask", "Overlay"])):
        axes[row_i, col_i].imshow(panel)
        axes[row_i, col_i].axis("off")
        axes[row_i, col_i].grid(False)
        axes[row_i, col_i].set_title(f"{class_name} - {title}" if col_i == 0 else title, fontsize=13)
fig.suptitle("SegFormer - Dataset Examples (one per class)", fontsize=15, fontweight="bold", y=1.0)
fig.tight_layout()
fig.savefig(FIG / "dataset_examples.png", dpi=120)
plt.show()


In [ ]:
# ============================================================
# CELL 7 — DATA PREPROCESSING FOR SEGFORMER (512 crop, ImageNet normalisation)
# Purpose: Build train / val / test datasets and loaders ONCE.
# Train : random crop -> 512x512, random horizontal / vertical flip.
# Val/Test: whole image resized to 512x512 (no augmentation).
# Masks are class-index images (0..6), resized with nearest neighbour only.
# Self-supervised views are defined inside each method (cells for DINOv3 / BYOL / MAE).
# ============================================================
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def normalize_to_tensor(rgb_uint8):
    """(H, W, 3) uint8 -> (3, H, W) float tensor, ImageNet mean/std."""
    x = (rgb_uint8.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))


def preprocess_rgb(rgb_uint8):
    """Evaluation-time preprocessing: resize to 512x512 + normalise."""
    resized = cv2.resize(rgb_uint8, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    return normalize_to_tensor(resized)


def denormalize(x):
    """(3, H, W) tensor -> (H, W, 3) uint8 for display."""
    arr = x.detach().cpu().numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
    return (np.clip(arr, 0, 1) * 255).astype(np.uint8)


def random_crop_resize(img, mask):
    h, w = mask.shape
    for _ in range(10):
        area = h * w * np.random.uniform(0.5, 1.0)
        ratio = np.exp(np.random.uniform(np.log(3 / 4), np.log(4 / 3)))
        cw, ch = int(round(np.sqrt(area * ratio))), int(round(np.sqrt(area / ratio)))
        if 0 < cw <= w and 0 < ch <= h:
            x0, y0 = np.random.randint(0, w - cw + 1), np.random.randint(0, h - ch + 1)
            img, mask = img[y0:y0 + ch, x0:x0 + cw], mask[y0:y0 + ch, x0:x0 + cw]
            break
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return img, mask


class SegDataset(Dataset):
    def __init__(self, split, train):
        self.split, self.train = split, train
        self.items = sorted(IMAGE_INDEX[split].items())

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        uid, path = self.items[index]
        rgb, gt = load_rgb(path), load_gt(self.split, uid)

        if self.train:
            rgb, mask = random_crop_resize(rgb, gt)
            if np.random.rand() < 0.5:
                rgb, mask = rgb[:, ::-1], mask[:, ::-1]
            if np.random.rand() < 0.5:
                rgb, mask = rgb[::-1], mask[::-1]
            return (normalize_to_tensor(np.ascontiguousarray(rgb)),
                    torch.from_numpy(np.ascontiguousarray(mask)).long())

        mask512 = cv2.resize(gt, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        return (preprocess_rgb(rgb), torch.from_numpy(mask512).long(), gt, uid)


def collate_eval(batch):
    xs, ys, gts, uids = zip(*batch)
    return torch.stack(xs), torch.stack(ys), list(gts), list(uids)


def worker_init(worker_id):
    np.random.seed((torch.initial_seed() + worker_id) % 2 ** 32)


_generator = torch.Generator()
_generator.manual_seed(SEED)

train_set = SegDataset("train", train=True)
val_set = SegDataset("val", train=False)
test_set = SegDataset("test", train=False)

train_loader = DataLoader(train_set, batch_size=FT_BATCH, shuffle=True, drop_last=True,
                          num_workers=WORKERS, pin_memory=(DEVICE == "cuda"),
                          worker_init_fn=worker_init, generator=_generator)
val_loader = DataLoader(val_set, batch_size=FT_BATCH, shuffle=False, num_workers=WORKERS,
                        collate_fn=collate_eval)
test_loader = DataLoader(test_set, batch_size=FT_BATCH, shuffle=False, num_workers=WORKERS,
                         collate_fn=collate_eval)

print("train / val / test images:", len(train_set), len(val_set), len(test_set))
print("train batches per epoch  :", len(train_loader), f"(batch size {FT_BATCH})")


In [ ]:
# ============================================================
# CELL 8 — PREPROCESSING CHECK: one training batch, masks must line up
# Purpose: Draw one batch after preprocessing and verify image/mask alignment.
# ============================================================
x_batch, y_batch = next(iter(train_loader))
print("images:", tuple(x_batch.shape), x_batch.dtype, "| masks:", tuple(y_batch.shape), y_batch.dtype)
print("mask values in batch:", sorted(torch.unique(y_batch).tolist()))
print("image mean/std after normalisation: %.2f / %.2f" % (x_batch.mean().item(), x_batch.std().item()))

assert x_batch.shape[-2:] == (IMG_SIZE, IMG_SIZE) and y_batch.shape[-2:] == (IMG_SIZE, IMG_SIZE)
assert int(y_batch.min()) >= 0 and int(y_batch.max()) <= NC

n_show = min(4, x_batch.shape[0])
fig, axes = plt.subplots(2, n_show, figsize=(4.2 * n_show, 8.4), squeeze=False)
for j in range(n_show):
    img, msk = denormalize(x_batch[j]), y_batch[j].numpy()
    axes[0, j].imshow(img)
    overlay = np.where((msk > 0)[..., None], (img * 0.5 + color_mask(msk) * 0.5).astype(np.uint8), img)
    axes[1, j].imshow(overlay)
    axes[0, j].set_title("Augmented image" if j == 0 else "")
    axes[1, j].set_title("Image + mask overlay" if j == 0 else "")
    for r in range(2):
        axes[r, j].axis("off"); axes[r, j].grid(False)
fig.suptitle("SegFormer - Preprocessed Training Batch", fontsize=15, fontweight="bold", y=1.0)
fig.tight_layout()
fig.savefig(FIG / "preprocessing_batch_check.png", dpi=120)
plt.show()
del x_batch, y_batch


In [ ]:
# ============================================================
# CELL 9 — FIXED 12-IMAGE TEST REFERENCE SET (Step 2)
# Purpose: Lock a constant list of 12 test images (2 per disease class), save it,
#          and reuse the same list in every figure that uses it.
# Saves: reference_12.json
# ============================================================
REF_PATH = EV / "reference_12.json"

if REF_PATH.exists():
    REFERENCE_12 = json.loads(REF_PATH.read_text())
else:
    _test = man[man["split"] == "test"]
    REFERENCE_12 = []
    for cname in CLASS_NAMES:
        _rows = _test[_test["class_name"] == cname].sort_values("uid")
        REFERENCE_12 += [{"uid": u, "class": cname} for u in _rows["uid"].head(2)]
    assert len(REFERENCE_12) == 12, f"Expected 12, got {len(REFERENCE_12)}"
    REF_PATH.write_text(json.dumps(REFERENCE_12, indent=2))

print("Reference set locked:", [r["uid"] for r in REFERENCE_12])

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, ref in zip(axes.flat, REFERENCE_12):
    rgb = load_rgb(IMAGE_INDEX["test"][ref["uid"]])
    gt = load_gt("test", ref["uid"])
    ax.imshow(np.where((gt > 0)[..., None], (rgb * 0.5 + color_mask(gt) * 0.5).astype(np.uint8), rgb))
    ax.set_title(ref["class"], fontsize=14)
    ax.axis("off"); ax.grid(False)
fig.suptitle("SegFormer - Fixed 12-Image Test Reference Set (ground truth)", fontsize=15,
             fontweight="bold", y=1.0)
fig.tight_layout()
fig.savefig(FIG / "reference12_ground_truth.png", dpi=110)
plt.show()


In [ ]:
# ============================================================
# CELL 10 — SEGFORMER (MiT-B2) ARCHITECTURE: ENCODER + SEGMENTATION HEAD
# Purpose: Define the model used by ALL four runs (DINOv3, BYOL, MAE, baseline) and print parameter counts.
# Recipe: MiT-B2 encoder (4 stages, depths 3-4-6-3) + all-MLP decode head (768 ch), 7 classes.
# Baseline start weights: BASELINE_INIT (cell 2). "random" mirrors the YOLO26s baseline
# (yolo26s-seg.yaml, pretrained=False). SSL runs replace ONLY the encoder weights; the decode head
# always starts from the same random initialisation (same seed) in all four runs.
# ============================================================

def build_config():
    return SegformerConfig(
        num_encoder_blocks=4,
        depths=[3, 4, 6, 3],
        sr_ratios=[8, 4, 2, 1],
        hidden_sizes=[64, 128, 320, 512],
        patch_sizes=[7, 3, 3, 3],
        strides=[4, 2, 2, 2],
        num_attention_heads=[1, 2, 5, 8],
        mlp_ratios=[4, 4, 4, 4],
        decoder_hidden_size=768,
        num_labels=NUM_CLASSES,
        id2label={i: MASK_NAMES[i] for i in range(NUM_CLASSES)},
        label2id={MASK_NAMES[i]: i for i in range(NUM_CLASSES)},
    )


def new_model(init=None):
    """Fresh baseline model for training."""
    init = BASELINE_INIT if init is None else init
    if init == "imagenet":
        return SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b2",
            num_labels=NUM_CLASSES,
            id2label={i: MASK_NAMES[i] for i in range(NUM_CLASSES)},
            label2id={MASK_NAMES[i]: i for i in range(NUM_CLASSES)},
            ignore_mismatched_sizes=True,
        )
    assert init == "random", init
    return SegformerForSemanticSegmentation(build_config())


def model_shell():
    """Empty model with the same architecture, used to load saved checkpoints."""
    return SegformerForSemanticSegmentation(build_config())


import torchvision.transforms as T
from transformers import SegformerModel

IMNET_MEAN_T = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
IMNET_STD_T = torch.tensor(IMAGENET_STD, device=DEVICE).view(1, 3, 1, 1)


class MiTBackbone(nn.Module):
    """MiT-B2 encoder for self-supervised pretraining. Input: RGB in [0, 1]; ImageNet normalisation
    is applied inside. Output: global-average-pooled last-stage feature (512-d).
    Its state_dict keys are identical to model.segformer.state_dict() of the segmentation model."""
    feature_dim = 512

    def __init__(self):
        super().__init__()
        self.encoder = SegformerModel(build_config())

    def forward(self, x01):
        x = (x01 - IMNET_MEAN_T) / IMNET_STD_T
        return self.encoder(pixel_values=x).last_hidden_state.mean(dim=(2, 3))

    def state(self):
        return {k: v.detach().cpu().clone() for k, v in self.encoder.state_dict().items()}


def count_parameters(model):
    encoder = sum(p.numel() for p in model.segformer.parameters())
    head = sum(p.numel() for p in model.decode_head.parameters())
    return encoder, head, sum(p.numel() for p in model.parameters())


seed_all(SEED)
probe = new_model().eval()
enc_p, head_p, all_p = count_parameters(probe)
print(f"Encoder (MiT-B2) parameters : {enc_p / 1e6:7.3f} M")
print(f"Decode head parameters      : {head_p / 1e6:7.3f} M")
print(f"Whole model parameters      : {all_p / 1e6:7.3f} M")

with torch.no_grad():
    probe_out = probe(pixel_values=torch.zeros(1, 3, IMG_SIZE, IMG_SIZE), output_hidden_states=True)
print("Logits shape (before upsampling):", tuple(probe_out.logits.shape))
print("Encoder stage shapes:", [tuple(h.shape[1:]) for h in probe_out.hidden_states])
print("Baseline start weights:", BASELINE_INIT)
del probe, probe_out
cleanup_gpu()


In [ ]:
# ============================================================
# CELL 11 — SHARED SSL DATASET AND AUGMENTATION UTILITIES
# Purpose: Load TRAINING images without labels for self-supervised pretraining.
# Validation and test images are never read here (SSL_FILES = train split only).
# ============================================================
import copy
from sklearn.neighbors import KNeighborsClassifier

SSL_FILES = [line.strip() for line in (DS / "ssl_train_images.txt").read_text().splitlines() if line.strip()]
assert len(SSL_FILES) == EXPECTED_COUNTS["train"]
assert all("/images/train/" in p.replace("\\", "/") for p in SSL_FILES)


class SSLDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths, self.transform = paths, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as im:
            return self.transform(im.convert("RGB"))


def make_loader(transform, batch):
    return DataLoader(SSLDataset(SSL_FILES, transform), batch_size=batch, shuffle=True, drop_last=True,
                      num_workers=WORKERS, pin_memory=(DEVICE == "cuda"))


def base_augmentation(size, scale):
    return T.Compose([
        T.RandomResizedCrop(size, scale=scale),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomApply([T.ColorJitter(.4, .4, .2, .05)], p=.8),
        T.RandomGrayscale(p=.1),
        T.ToTensor(),
    ])


def teacher_augmentation():
    return T.Compose([T.RandomResizedCrop(SSL_SIZE, scale=(.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor()])


def make_mlp(input_dim, hidden_dim, output_dim):
    return nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim),
                         nn.ReLU(inplace=True), nn.Linear(hidden_dim, output_dim))


def epoch_lr(epoch):
    """Cosine decay across SSL_EPOCHS epochs (BYOL)."""
    progress = epoch / max(SSL_EPOCHS - 1, 1)
    return 1e-5 + .5 * (.001 - 1e-5) * (1 + math.cos(math.pi * progress))


def save_ssl_history(name, history):
    pd.DataFrame(history).to_csv(LOGS / f"{name}_segformer_ssl_history.csv", index=False)


def save_resume(path, modules, optimizer, scaler, epoch, history):
    torch.save({**{n: m.state_dict() for n, m in modules.items()},
                "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                "epoch": epoch, "history": history}, path)


def load_resume(path, modules, optimizer, scaler):
    if not path.exists():
        return 0, []
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    for n, m in modules.items():
        m.load_state_dict(ck[n])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    print(f"Resuming from epoch {ck['epoch']}: {path.name}")
    return ck["epoch"], ck["history"]


def clip_and_step(loss, params, optimizer, scaler, max_norm):
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(params, max_norm)
    scaler.step(optimizer)
    scaler.update()


class GlobalDistillStudent(nn.Module):
    """MiT-B2 encoder -> global feature -> projector -> L2-normalised vector (matches a frozen teacher)."""
    def __init__(self, backbone, teacher_dim, hidden=512):
        super().__init__()
        self.backbone = backbone
        self.projector = nn.Sequential(nn.Linear(MiTBackbone.feature_dim, hidden), nn.GELU(),
                                       nn.Linear(hidden, teacher_dim))

    def forward(self, x01):
        return F.normalize(self.projector(self.backbone(x01)).float(), dim=-1)


print("SSL data utilities ready. SSL training images:", len(SSL_FILES))
print("SSL epochs:", SSL_EPOCHS, "| SSL crop:", SSL_SIZE, "| BYOL batch:", SSL_BATCH, "| teacher batch:", TEACHER_BATCH)
print("BYOL learning rates per epoch:", [round(epoch_lr(e), 7) for e in range(SSL_EPOCHS)][:5], "...")


In [ ]:
import transformers, huggingface_hub
print("transformers:", transformers.__version__, "| hub:", huggingface_hub.__version__)
from transformers import SegformerConfig, SegformerForSemanticSegmentation, AutoConfig
print("SegFormer import OK")

In [ ]:
# ============================================================
# CELL 12 — HUGGING FACE AUTHENTICATION FOR DINOv3
# Skipped automatically when all three SSL encoders are already finished (DINOv3 is not needed then).
# ============================================================
import os

DINO_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"
MAE_ID = "facebook/vit-mae-base"
HF_TOKEN = None

_ssl_finals = ["dino_segformer_backbone_final.pt", "byol_segformer_backbone_final.pt",
               "mae_segformer_backbone_final.pt"]
if all((CKPT / f).exists() for f in _ssl_finals):
    print("All SSL pretraining is already finished -> DINOv3 teacher not needed, token check skipped.")
else:
    from huggingface_hub import HfApi
    from transformers import AutoConfig
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        from google.colab import userdata
        for _label in ("HF_TOKEN", "Colab_token", "COLAB_TOKEN"):
            try:
                HF_TOKEN = userdata.get(_label)
            except Exception:
                HF_TOKEN = None
            if HF_TOKEN:
                break
    assert HF_TOKEN, "HF token not found: add a Colab secret named HF_TOKEN with notebook access ON."
    print("Authenticated Hugging Face account:", HfApi(token=HF_TOKEN).whoami()["name"])
    _cfg = AutoConfig.from_pretrained(DINO_ID, token=HF_TOKEN)      # shows the REAL error if it fails
    print("DINOv3 access verified:", _cfg.model_type)

In [ ]:
# ============================================================
# CELL 13 — DINOv3 -> SEGFORMER (MiT-B2): DEFINE ARCHITECTURE + SSL TRAINING (Stage A)
# Recipe (same as the YOLO26s notebook): frozen pretrained DINOv3 ViT-S/16 teacher; the MiT-B2 student
# (+ projector) is trained so that its normalised global vector matches the teacher CLS/pooled vector.
# Loss = 1 - cosine similarity | AdamW lr 3e-4, wd 1e-4 | crop 224 (scale .6-1) + h-flip | grad-clip 2.
# Training images only. Steps: SSL_EPOCHS. Resumes from the last finished epoch if interrupted.
# ============================================================
from transformers import AutoModel

DINO_FINAL = CKPT / "dino_segformer_backbone_final.pt"
DINO_LAST = CKPT / "dino_segformer_last.pt"
DINO_LR, DINO_WD, DINO_CLIP = 3e-4, 1e-4, 2.0

print("DINOv3 recipe: teacher", DINO_ID, "| crop", SSL_SIZE, "| batch", TEACHER_BATCH,
      "| lr", DINO_LR, "| wd", DINO_WD, "| loss 1-cosine")

if DINO_FINAL.exists():
    ck = torch.load(DINO_FINAL, map_location="cpu", weights_only=False)
    assert ck["method"] == "pretrained_dinov3_distillation" and ck["epochs"] == SSL_EPOCHS
    print("Existing DINOv3 checkpoint found, skipping training:", DINO_FINAL)
else:
    seed_all(SEED)
    dino_teacher = AutoModel.from_pretrained(DINO_ID, token=HF_TOKEN, attn_implementation="sdpa").to(DEVICE).eval()
    dino_teacher.requires_grad_(False)
    teacher_dim = dino_teacher.config.hidden_size
    print("DINOv3 teacher loaded. Hidden size:", teacher_dim)

    backbone = MiTBackbone().to(DEVICE)
    student = GlobalDistillStudent(backbone, teacher_dim).to(DEVICE)
    print("Student: encoder %.2f M | projector %.2f M parameters" % (
        sum(p.numel() for p in backbone.parameters()) / 1e6,
        sum(p.numel() for p in student.projector.parameters()) / 1e6))

    optimizer = torch.optim.AdamW(student.parameters(), lr=DINO_LR, weight_decay=DINO_WD)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    loader = make_loader(teacher_augmentation(), TEACHER_BATCH)
    start_epoch, history = load_resume(DINO_LAST, {"student": student}, optimizer, scaler)

    for epoch in range(start_epoch, SSL_EPOCHS):
        _t0 = time.time()
        student.train()
        losses = []
        for images in tqdm(loader, desc=f"DINOv3 distillation {epoch + 1}/{SSL_EPOCHS}"):
            images = images.to(DEVICE)
            with torch.no_grad():
                out = dino_teacher(pixel_values=(images - IMNET_MEAN_T) / IMNET_STD_T)
                target = out.pooler_output if getattr(out, "pooler_output", None) is not None \
                    else out.last_hidden_state[:, 0, :]
                target = F.normalize(target.float(), dim=-1)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                prediction = student(images)
            loss = 1 - (prediction.float() * target).sum(dim=-1).mean()
            clip_and_step(loss, student.parameters(), optimizer, scaler, DINO_CLIP)
            losses.append(float(loss.item()))

        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)),
                        "objective": "dinov3_teacher_cosine_distillation"})
        save_ssl_history("dino", history)
        save_resume(DINO_LAST, {"student": student}, optimizer, scaler, epoch + 1, history)
        print(f"DINOv3 epoch {epoch + 1}/{SSL_EPOCHS} | loss {history[-1]['loss']:.4f}")
        if epoch + 1 < SSL_EPOCHS and time_up(time.time() - _t0):
            break

    if STOP_TRAINING:
        halt_notebook("DINOv3 SSL pretraining")
    else:
        torch.save({"encoder": backbone.state(), "method": "pretrained_dinov3_distillation",
                    "teacher_id": DINO_ID, "epochs": SSL_EPOCHS}, DINO_FINAL)
        print("DINOv3 pretraining completed:", DINO_FINAL)
    del dino_teacher, student, backbone, optimizer, loader, scaler
    cleanup_gpu()


In [ ]:
# ============================================================
# CELL 14 — BYOL -> SEGFORMER (MiT-B2): DEFINE ARCHITECTURE + SSL TRAINING (Stage A)
# Recipe (same as the YOLO26s notebook): online network (encoder + projector 512-4096-256 + predictor)
# predicts the projection of a target network (EMA, momentum .996 -> 1, NO negative pairs).
# Two views per image: RandomResizedCrop 224 (scale .4-1), flips, colour jitter, grayscale.
# Loss = byol_loss(p1, t2) + byol_loss(p2, t1) | AdamW wd .04 | cosine lr .001 -> 1e-5 | grad-clip 3.
# BYOL_EXPORT: which encoder is saved as SSL weights. "online" (default) is the standard BYOL choice
# and is the more informative one when only a few epochs are run; the YOLO26s notebook saved "target".
# ============================================================
BYOL_EXPORT = "online"                 # "online" or "target"
BYOL_FINAL = CKPT / "byol_segformer_backbone_final.pt"
BYOL_LAST = CKPT / "byol_segformer_last.pt"


class BYOLViews:
    def __init__(self):
        self.augment = base_augmentation(SSL_SIZE, (.4, 1.0))

    def __call__(self, image):
        return [self.augment(image), self.augment(image)]


class BYOLNetwork(nn.Module):
    def __init__(self, backbone, predictor=True):
        super().__init__()
        self.backbone = backbone
        self.projector = make_mlp(MiTBackbone.feature_dim, 4096, 256)
        self.predictor = make_mlp(256, 4096, 256) if predictor else None

    def forward(self, x01):
        projection = self.projector(self.backbone(x01))
        return projection, (self.predictor(projection) if self.predictor is not None else None)


def byol_loss(prediction, target):
    return 2 - 2 * F.cosine_similarity(prediction.float(), target.float().detach(), dim=-1).mean()


print("BYOL recipe: views 2 x crop", SSL_SIZE, "| batch", SSL_BATCH, "| lr cosine .001->1e-5 | wd .04 | EMA .996->1",
      "| export:", BYOL_EXPORT)

if BYOL_FINAL.exists():
    ck = torch.load(BYOL_FINAL, map_location="cpu", weights_only=False)
    assert ck["method"] == "byol" and ck["epochs"] == SSL_EPOCHS
    print("Existing BYOL checkpoint found, skipping training:", BYOL_FINAL)
else:
    seed_all(SEED)
    online = BYOLNetwork(MiTBackbone(), predictor=True).to(DEVICE)
    target = BYOLNetwork(MiTBackbone(), predictor=False).to(DEVICE)
    target.backbone.load_state_dict(online.backbone.state_dict())
    target.projector.load_state_dict(online.projector.state_dict())
    for p in target.parameters():
        p.requires_grad = False
    print("Online: encoder %.2f M | whole BYOL net %.2f M parameters" % (
        sum(p.numel() for p in online.backbone.parameters()) / 1e6,
        sum(p.numel() for p in online.parameters()) / 1e6))

    optimizer = torch.optim.AdamW(online.parameters(), lr=.001, weight_decay=.04)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    loader = make_loader(BYOLViews(), SSL_BATCH)
    start_epoch, history = load_resume(BYOL_LAST, {"online": online, "target": target}, optimizer, scaler)

    for epoch in range(start_epoch, SSL_EPOCHS):
        _t0 = time.time()
        online.train()
        target.train()
        lr = epoch_lr(epoch)
        for group in optimizer.param_groups:
            group["lr"] = lr
        losses = []

        for views in tqdm(loader, desc=f"BYOL epoch {epoch + 1}/{SSL_EPOCHS}"):
            v1, v2 = [x.to(DEVICE) for x in views]
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                _, p1 = online(v1)
                _, p2 = online(v2)
                with torch.no_grad():
                    t1, _ = target(v1)
                    t2, _ = target(v2)
                loss = byol_loss(p1, t2) + byol_loss(p2, t1)
            clip_and_step(loss, online.parameters(), optimizer, scaler, 3.0)

            momentum = .996 + (1 - .996) * (epoch / SSL_EPOCHS)
            with torch.no_grad():
                for o_mod, t_mod in [(online.backbone, target.backbone), (online.projector, target.projector)]:
                    for op, tp in zip(o_mod.parameters(), t_mod.parameters()):
                        tp.mul_(momentum).add_(op, alpha=1 - momentum)
            losses.append(float(loss.item()))

        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)), "lr": lr})
        save_ssl_history("byol", history)
        save_resume(BYOL_LAST, {"online": online, "target": target}, optimizer, scaler, epoch + 1, history)
        print(f"BYOL epoch {epoch + 1}/{SSL_EPOCHS} | loss {history[-1]['loss']:.4f}")
        if epoch + 1 < SSL_EPOCHS and time_up(time.time() - _t0):
            break

    if STOP_TRAINING:
        halt_notebook("BYOL SSL pretraining")
    else:
        exported = online.backbone if BYOL_EXPORT == "online" else target.backbone
        torch.save({"encoder": exported.state(), "method": "byol", "epochs": SSL_EPOCHS,
                    "exported_branch": BYOL_EXPORT}, BYOL_FINAL)
        print("BYOL pretraining completed:", BYOL_FINAL)
    del online, target, optimizer, loader, scaler
    cleanup_gpu()


In [ ]:
# ============================================================
# CELL 15 — MAE -> SEGFORMER (MiT-B2): DEFINE ARCHITECTURE + SSL TRAINING (Stage A)
# Recipe (same as the YOLO26s notebook): standard ViT-MAE (facebook/vit-mae-base: encoder + light
# Transformer decoder + random patch masking, 75 % of patches hidden by default).
#   Step A: SSL_EPOCHS epochs of masked-patch reconstruction (pixel MSE on hidden patches) on the
#           training images, AdamW lr 1e-5, wd .05 (continued MAE training on papaya leaves).
#   Step B: SSL_EPOCHS epochs: the MAE encoder CLS vector (mask ratio 0, full image) is distilled into
#           the MiT-B2 student (1 - cosine, AdamW lr 3e-4, wd 1e-4). MiT-B2 is hierarchical, so it is
#           not trained with patch masks directly; it receives the MAE encoder's knowledge.
# Training images only. No token is needed (public model).
# ============================================================
try:
    from transformers import ViTMAEForPreTraining
except ImportError as e:
    print(f"Warning: Could not import ViTMAEForPreTraining directly due to a library issue: {e}")
    ViTMAEForPreTraining = None

MAE_FINAL = CKPT / "mae_segformer_backbone_final.pt"
MAE_TEACHER = CKPT / "mae_vit_teacher.pt"
MAE_LAST = CKPT / "mae_segformer_last.pt"
MAE_TEACHER_LAST = CKPT / "mae_vit_teacher_last.pt"      # Step A resume file (per epoch)

print("MAE recipe: teacher", MAE_ID, "| crop", SSL_SIZE, "| batch", TEACHER_BATCH,
      "| Step A lr 1e-5 wd .05 | Step B lr 3e-4 wd 1e-4 | loss 1-cosine")

if MAE_FINAL.exists():
    ck = torch.load(MAE_FINAL, map_location="cpu", weights_only=False)
    assert ck["method"] == "standard_vit_mae_then_segformer_distillation" and ck["epochs"] == SSL_EPOCHS
    print("Existing MAE checkpoint found, skipping training:", MAE_FINAL)
else:
    if ViTMAEForPreTraining is None:
        raise ImportError("ViTMAEForPreTraining is required to train MAE from scratch, but import failed.")
    seed_all(SEED)
    mae_loader = make_loader(teacher_augmentation(), TEACHER_BATCH)
    mae_teacher = ViTMAEForPreTraining.from_pretrained(MAE_ID).to(DEVICE)
    print("MAE mask ratio:", mae_teacher.config.mask_ratio, "| hidden size:", mae_teacher.config.hidden_size)

    # ---- Step A: masked-patch reconstruction ----
    if MAE_TEACHER.exists():
        mae_teacher.load_state_dict(torch.load(MAE_TEACHER, map_location="cpu", weights_only=True))
        print("Restored finished MAE reconstruction stage:", MAE_TEACHER)
    else:
        opt_a = torch.optim.AdamW(mae_teacher.parameters(), lr=1e-5, weight_decay=.05)
        recon_history, start_a = [], 0
        if MAE_TEACHER_LAST.exists():
            ck = torch.load(MAE_TEACHER_LAST, map_location=DEVICE, weights_only=False)
            mae_teacher.load_state_dict(ck["model"])
            opt_a.load_state_dict(ck["optimizer"])
            recon_history, start_a = ck["history"], ck["epoch"]
            print("Resuming MAE reconstruction from epoch", start_a)
        for epoch in range(start_a, SSL_EPOCHS):
            _t0 = time.time()
            mae_teacher.train()
            losses = []
            for images in tqdm(mae_loader, desc=f"MAE reconstruction {epoch + 1}/{SSL_EPOCHS}"):
                images = images.to(DEVICE)
                loss = mae_teacher(pixel_values=(images - IMNET_MEAN_T) / IMNET_STD_T).loss
                opt_a.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(mae_teacher.parameters(), 1.0)
                opt_a.step()
                losses.append(float(loss.item()))
            recon_history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)),
                                  "objective": "standard_mae_masked_patch_mse"})
            save_ssl_history("mae_reconstruction", recon_history)
            torch.save({"model": mae_teacher.state_dict(), "optimizer": opt_a.state_dict(),
                        "epoch": epoch + 1, "history": recon_history}, MAE_TEACHER_LAST)
            print(f"MAE reconstruction epoch {epoch + 1}/{SSL_EPOCHS} | loss {recon_history[-1]['loss']:.4f}")
            if epoch + 1 < SSL_EPOCHS and time_up(time.time() - _t0):
                break
        if not STOP_TRAINING:
            torch.save({k: v.cpu() for k, v in mae_teacher.state_dict().items()}, MAE_TEACHER)
        del opt_a
        cleanup_gpu()

    # ---- Step B: distil the MAE encoder into MiT-B2 ----
    mae_teacher.eval().requires_grad_(False)
    old_mask_ratio = mae_teacher.config.mask_ratio
    mae_teacher.config.mask_ratio = 0.0

    backbone = MiTBackbone().to(DEVICE)
    student = GlobalDistillStudent(backbone, mae_teacher.config.hidden_size).to(DEVICE)
    optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    start_epoch, history = load_resume(MAE_LAST, {"student": student}, optimizer, scaler)

    for epoch in range(start_epoch, 0 if STOP_TRAINING else SSL_EPOCHS):   # skip if Step A stopped early
        _t0 = time.time()
        student.train()
        losses = []
        for images in tqdm(mae_loader, desc=f"MAE -> MiT-B2 distillation {epoch + 1}/{SSL_EPOCHS}"):
            images = images.to(DEVICE)
            with torch.no_grad():
                target = mae_teacher.vit(pixel_values=(images - IMNET_MEAN_T) / IMNET_STD_T).last_hidden_state[:, 0]
                target = F.normalize(target.float(), dim=-1)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                prediction = student(images)
            loss = 1 - (prediction.float() * target).sum(dim=-1).mean()
            clip_and_step(loss, student.parameters(), optimizer, scaler, 2.0)
            losses.append(float(loss.item()))
        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)),
                        "objective": "mae_encoder_cosine_distillation"})
        save_ssl_history("mae", history)
        save_resume(MAE_LAST, {"student": student}, optimizer, scaler, epoch + 1, history)
        print(f"MAE transfer epoch {epoch + 1}/{SSL_EPOCHS} | loss {history[-1]['loss']:.4f}")
        if epoch + 1 < SSL_EPOCHS and time_up(time.time() - _t0):
            break

    mae_teacher.config.mask_ratio = old_mask_ratio
    if STOP_TRAINING:
        halt_notebook("MAE SSL pretraining")
    else:
        torch.save({"encoder": backbone.state(), "method": "standard_vit_mae_then_segformer_distillation",
                    "teacher_id": MAE_ID, "epochs": SSL_EPOCHS, "mae_reconstruction_epochs": SSL_EPOCHS}, MAE_FINAL)
        print("MAE pretraining completed:", MAE_FINAL)
    del mae_teacher, student, backbone, optimizer, mae_loader, scaler
    cleanup_gpu()


In [ ]:
# CELL 16 — SSL pretraining loss: DINOv3 (SegFormer MiT-B2 student)
h = pd.read_csv(LOGS / "dino_segformer_ssl_history.csv")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h["epoch"], h["loss"], marker="o", linewidth=2, color=MODEL_COLORS["dino"])
ax.set(xlabel="SSL epoch", ylabel="Training objective (1 - cosine)")
ax.set_xticks(list(h["epoch"]) if len(h) <= 20 else [1] + list(range(10, len(h) + 1, 10)))
ax.set_title("SegFormer (DINOv3) - SSL Pretraining Loss", fontsize=13, fontweight="bold")
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "dino_ssl_loss_modelwise.png", dpi=150)
plt.show()


In [ ]:
# CELL 17 — SSL pretraining loss: BYOL (SegFormer MiT-B2 online network)
h = pd.read_csv(LOGS / "byol_segformer_ssl_history.csv")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h["epoch"], h["loss"], marker="o", linewidth=2, color=MODEL_COLORS["byol"])
ax.set(xlabel="SSL epoch", ylabel="Training objective (BYOL loss)")
ax.set_xticks(list(h["epoch"]) if len(h) <= 20 else [1] + list(range(10, len(h) + 1, 10)))
ax.set_title("SegFormer (BYOL) - SSL Pretraining Loss", fontsize=13, fontweight="bold")
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "byol_ssl_loss_modelwise.png", dpi=150)
plt.show()


In [ ]:
# CELL 18 — SSL pretraining loss: MAE (Step A reconstruction, Step B transfer into MiT-B2)
ha = pd.read_csv(LOGS / "mae_reconstruction_segformer_ssl_history.csv")
hb = pd.read_csv(LOGS / "mae_segformer_ssl_history.csv")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ha["epoch"], ha["loss"], marker="o", linewidth=2, color=MODEL_COLORS["mae"])
axes[0].set(xlabel="SSL epoch", ylabel="Masked-patch MSE", title="Step A: MAE reconstruction")
axes[1].plot(hb["epoch"], hb["loss"], marker="o", linewidth=2, color=MODEL_COLORS["mae"])
axes[1].set(xlabel="SSL epoch", ylabel="1 - cosine", title="Step B: transfer into MiT-B2")
for ax, h in zip(axes, (ha, hb)):
    ax.set_xticks(list(h["epoch"]) if len(h) <= 20 else [1] + list(range(10, len(h) + 1, 10)))
    ax.grid(alpha=.3)
fig.suptitle("SegFormer (MAE) - SSL Pretraining Loss", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "mae_ssl_loss_modelwise.png", dpi=150)
plt.show()


In [ ]:
# ============================================================
# CELL 19 — SSL HEALTH CHECKS + ssl_weights.pt
# Purpose: KNN probe on FROZEN features (fit on train, scored on val), collapse indicators
#          (feature spread, effective rank), loss trend. Saves <method>/ssl_weights.pt.
# Uses train and val images only (the test split is never touched here).
# MAE extra check: hidden-patch reconstruction error against a predict-the-mean baseline.
# ============================================================
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

SSL_CKPTS = {"dino": DINO_FINAL, "byol": BYOL_FINAL, "mae": MAE_FINAL}
_name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}


def split_items(split):
    """Sorted (path, class index 0..5) list for one split."""
    lab = dict(zip(man["uid"], man["class_name"]))
    paths = sorted(p for p in (DS / "images" / split).iterdir() if p.is_file())
    return paths, np.array([_name_to_idx[lab[p.stem]] for p in paths])


@torch.no_grad()
def extract_embeddings(backbone, paths, size=None, batch=16):
    size = size or IMG_SIZE
    backbone = backbone.to(DEVICE).eval()
    feats = []
    for i in range(0, len(paths), batch):
        imgs = [np.asarray(Image.open(p).convert("RGB").resize((size, size))) for p in paths[i:i + batch]]
        x = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float().to(DEVICE) / 255.0
        feats.append(backbone(x).float().cpu().numpy())
    return np.concatenate(feats)


def collapse_stats(E):
    En = E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)
    sv = np.linalg.svd(En - En.mean(0), compute_uv=False)
    p = sv ** 2 / max((sv ** 2).sum(), 1e-12)
    return {"mean_feature_std": float(En.std(axis=0).mean()),
            "effective_rank": float(np.exp(-(p * np.log(p + 1e-12)).sum())),
            "ideal_std_1_over_sqrt_D": float(1 / np.sqrt(E.shape[1]))}


def knn_probe(Etr, ytr, Eva, yva, k=20):
    clf = KNeighborsClassifier(n_neighbors=min(k, len(Etr)), metric="cosine").fit(Etr, ytr)
    return float((clf.predict(Eva) == yva).mean())


train_paths, y_train = split_items("train")
val_paths, y_val = split_items("val")

health_rows = []
for method in ["random_init"] + list(SSL_CKPTS):
    seed_all(SEED)
    bb = MiTBackbone()
    if method != "random_init":
        ck = torch.load(SSL_CKPTS[method], map_location="cpu", weights_only=False)
        bb.encoder.load_state_dict(ck["encoder"], strict=True)
        shutil.copy2(SSL_CKPTS[method], EXP_ROOTS[method] / "ssl_weights.pt")        # artifact: <method>/ssl_weights.pt
    Etr = extract_embeddings(bb, train_paths)
    Eva = extract_embeddings(bb, val_paths)
    row = {"encoder": method, "KNN_val_acc": knn_probe(Etr, y_train, Eva, y_val)}
    row.update(collapse_stats(Etr))
    if method != "random_init":
        h = pd.read_csv(LOGS / f"{method}_segformer_ssl_history.csv")
        row["SSL_loss_first"], row["SSL_loss_last"] = float(h["loss"].iloc[0]), float(h["loss"].iloc[-1])
        row["SSL_loss_trend"] = "decreasing" if h["loss"].iloc[-1] < h["loss"].iloc[0] else "NOT decreasing"
    health_rows.append(row)
    del bb
    cleanup_gpu()

health_df = pd.DataFrame(health_rows).set_index("encoder")
health_df.to_csv(EV / "ssl_health_checks.csv")
display(health_df.round(4))

chance = 1 / len(np.unique(y_train))
colors = ["#555555", MODEL_COLORS["dino"], MODEL_COLORS["byol"], MODEL_COLORS["mae"]]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
bars = axes[0].bar(health_df.index, health_df["KNN_val_acc"], color=colors, edgecolor="black", linewidth=1.2)
axes[0].axhline(chance, color="red", ls="--", lw=2, label=f"chance = {chance:.2f}")
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + .012, f"{bar.get_height():.3f}",
                 ha="center", fontsize=14, fontweight="bold")
axes[0].set_ylim(0, 1.1); axes[0].set_title("KNN probe accuracy (frozen features, val)")
axes[0].legend(); axes[0].grid(axis="x", visible=False)
bars = axes[1].bar(health_df.index, health_df["effective_rank"], color=colors, edgecolor="black", linewidth=1.2)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02, f"{bar.get_height():.1f}",
                 ha="center", fontsize=14, fontweight="bold")
axes[1].set_title("Effective rank of features (low = collapse risk)")
axes[1].grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIG / "ssl_health_checks.png")
plt.show()

# ---- MAE: hidden-patch reconstruction error vs predict-the-mean baseline (train images only) ----
try:
    chk = ViTMAEForPreTraining.from_pretrained(MAE_ID).to(DEVICE).eval()
    chk.load_state_dict(torch.load(MAE_TEACHER, map_location=DEVICE, weights_only=True))
    tf = T.Compose([T.Resize((SSL_SIZE, SSL_SIZE)), T.ToTensor()])
    sample = torch.stack([tf(Image.open(p).convert("RGB")) for p in train_paths[:64]]).to(DEVICE)
    with torch.no_grad():
        x = (sample - IMNET_MEAN_T) / IMNET_STD_T
        out = chk(pixel_values=x)
        tgt = chk.patchify(x)
        hidden = out.mask.bool()
        model_err = (out.logits - tgt).abs()[hidden].mean().item()
        mean_err = (tgt - tgt.mean()).abs()[hidden].mean().item()
    mae_check = pd.DataFrame([{"MAE_hidden_pixel_L1": model_err, "predict_the_mean_L1": mean_err,
                               "better_than_mean_baseline": model_err < mean_err}])
    mae_check.to_csv(EV / "mae_reconstruction_check.csv", index=False)
    display(mae_check.round(4))
    del chk
    cleanup_gpu()
except Exception as exc:
    print("MAE reconstruction check skipped:", repr(exc)[:200])


In [ ]:
# ============================================================
# CELL 20 — SEMANTIC SEGMENTATION EVALUATION UTILITIES
# Purpose: Confusion matrix, IoU, Dice, precision, recall, F1, and one function that
#          scores a whole loader. Same metric definitions as the YOLO26s notebook.
#          load_best_model(method) loads the best.pt of any of the four runs.
# Predictions are made at 512x512 and up-sampled to the ORIGINAL mask size, so
# the confusion matrices are counted on the original ground-truth pixels.
# ============================================================

def seg_loss(logits, target):
    """Pixel-wise cross-entropy after up-sampling the logits to the mask size."""
    up = F.interpolate(logits, size=target.shape[-2:], mode="bilinear", align_corners=False)
    return F.cross_entropy(up.float(), target)


def confusion_matrix(gt, pred):
    indices = gt.astype(np.int64) * NUM_CLASSES + pred.astype(np.int64)
    return np.bincount(indices.ravel(), minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)


def metrics_from_cm(cm):
    cm = cm.astype(float)
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    def safe_divide(numerator, denominator):
        return np.divide(numerator, denominator, out=np.full(numerator.shape, np.nan),
                         where=denominator > 0)

    iou = safe_divide(tp, tp + fp + fn)
    dice = safe_divide(2 * tp, 2 * tp + fp + fn)
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    f1 = safe_divide(2 * tp, 2 * tp + fp + fn)

    summary = {
        "mIoU_7": float(np.nanmean(iou)),
        "mIoU_fg6": float(np.nanmean(iou[1:])),
        "mDice_7": float(np.nanmean(dice)),
        "mDice_fg6": float(np.nanmean(dice[1:])),
        "mPrecision_fg6": float(np.nanmean(precision[1:])),
        "mRecall_fg6": float(np.nanmean(recall[1:])),
        "mF1_fg6": float(np.nanmean(f1[1:])),
        "pixel_accuracy": float(tp.sum() / max(cm.sum(), 1)),
    }
    per_class = pd.DataFrame(
        {"IoU": iou, "Dice": dice, "Precision": precision, "Recall": recall, "F1": f1},
        index=[MASK_NAMES[i] for i in range(NUM_CLASSES)],
    )
    return summary, per_class


def evaluate_loader(model, loader, save_dir=None):
    """Returns (per-image confusion matrices, uids, mean loss, images with no foreground)."""
    model.eval()
    cms, uids, loss_sum, no_prediction = [], [], 0.0, 0
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        for x, y512, gts, batch_uids in loader:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(pixel_values=x.to(DEVICE)).logits
            logits = logits.float()
            loss_sum += seg_loss(logits, y512.to(DEVICE)).item() * len(batch_uids)

            for j, uid in enumerate(batch_uids):
                h, w = gts[j].shape
                up = F.interpolate(logits[j:j + 1], size=(h, w), mode="bilinear", align_corners=False)
                pred = up.argmax(1)[0].cpu().numpy().astype(np.uint8)
                cms.append(confusion_matrix(gts[j], pred))
                uids.append(uid)
                if not (pred > 0).any():
                    no_prediction += 1
                if save_dir is not None:
                    Image.fromarray(pred).save(save_dir / f"{uid}.png")

    return np.asarray(cms, dtype=np.int64), uids, loss_sum / max(len(uids), 1), no_prediction


def load_best_model(method="baseline"):
    """Load the best-validation checkpoint (best.pt) of one run: baseline, dino, byol or mae."""
    checkpoint = torch.load(EXP_ROOTS[method] / "best.pt", map_location="cpu", weights_only=False)
    model = model_shell()
    model.load_state_dict(checkpoint["state_dict"])
    return model.to(DEVICE).eval()


@torch.no_grad()
def predict_image(model, rgb, return_probs=False):
    """One RGB image -> semantic mask at the original size (and optional class probabilities)."""
    x = preprocess_rgb(rgb).unsqueeze(0).to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
        logits = model(pixel_values=x).logits
    up = F.interpolate(logits.float(), size=rgb.shape[:2], mode="bilinear", align_corners=False)
    pred = up.argmax(1)[0].cpu().numpy().astype(np.uint8)
    return (pred, up.softmax(1)[0]) if return_probs else pred


print("Semantic evaluation functions initialized.")


In [ ]:
# ============================================================
# CELL 21 — SHARED FINE-TUNING UTILITIES — NO TRAINING IN THIS CELL
# Purpose: Train each of the four models for exactly FT_EPOCHS epochs, early stopping OFF,
#          log every epoch, keep the best-validation checkpoint (validation mIoU, 7 classes).
# Same seed, split, batch size, optimizer and learning rates for all four runs.
# Only the encoder start weights differ: baseline = BASELINE_INIT, DINOv3 / BYOL / MAE = <method>/ssl_weights.pt.
# Saves per run: history.csv, best.pt, last.pt (resume), DONE.json, init_check.json
# ============================================================
SSL_SOURCES = {"baseline": None,
               "dino": EXP_ROOTS["dino"] / "ssl_weights.pt",
               "byol": EXP_ROOTS["byol"] / "ssl_weights.pt",
               "mae": EXP_ROOTS["mae"] / "ssl_weights.pt"}

FT_CONFIG = {
    "epochs": FT_EPOCHS,
    "imgsz": IMG_SIZE,
    "batch": FT_BATCH,
    "optimizer": "AdamW",
    "lr_encoder": LR_ENCODER,
    "lr_head": LR_HEAD,
    "weight_decay": WEIGHT_DECAY,
    "schedule": "linear warm-up then linear decay (per iteration)",
    "warmup_fraction": WARMUP_FRACTION,
    "early_stopping": False,
    "seed": SEED,
    "amp": USE_AMP,
    "workers": WORKERS,
    "baseline_init": BASELINE_INIT,
    "ssl_epochs": SSL_EPOCHS,
    "checkpoint_rule": "highest validation mIoU over 7 classes",
}
(OUT / "ft_config.json").write_text(json.dumps(
    {"settings": FT_CONFIG, "sources": {k: str(v) if v else None for k, v in SSL_SOURCES.items()}}, indent=2))


def model_for(method):
    """Model for one run. Same seed -> identical decode-head start weights in all four runs."""
    seed_all(SEED)
    model = new_model(BASELINE_INIT if method == "baseline" else "random")
    info = {"method": method,
            "initialization": BASELINE_INIT if method == "baseline" else "ssl_encoder",
            "ssl_epochs": None if method == "baseline" else SSL_EPOCHS}
    if SSL_SOURCES[method] is not None:
        ck = torch.load(SSL_SOURCES[method], map_location="cpu", weights_only=False)
        assert ck["epochs"] == SSL_EPOCHS, "SSL checkpoint was trained for a different number of epochs"
        model.segformer.load_state_dict(ck["encoder"], strict=True)
        current = model.segformer.state_dict()
        identical = all(torch.equal(current[k].detach().cpu(), v) for k, v in ck["encoder"].items())
        assert identical, f"SSL transfer failed: {method}"
        info.update({"ssl_method": ck["method"], "transferred_tensors": len(ck["encoder"]),
                     "weights_identical": identical})
    return model, info


def train_experiment(method):
    root = EXP_ROOTS[method]
    best, last, done = root / "best.pt", root / "last.pt", root / "DONE.json"

    if done.exists() and best.exists():
        print("Already completed:", method)
        return str(best)

    model, info = model_for(method)
    model.to(DEVICE)

    encoder_params = [p for n, p in model.named_parameters() if n.startswith("segformer.")]
    head_params = [p for n, p in model.named_parameters() if n.startswith("decode_head.")]
    assert len(encoder_params) + len(head_params) == len(list(model.parameters()))

    optimizer = torch.optim.AdamW(
        [{"params": encoder_params, "lr": LR_ENCODER}, {"params": head_params, "lr": LR_HEAD}],
        weight_decay=WEIGHT_DECAY)
    total_iters = FT_EPOCHS * len(train_loader)
    warmup_iters = max(1, int(WARMUP_FRACTION * total_iters))

    def lr_factor(it):
        return min(1.0, (it + 1) / warmup_iters) * max(0.01, 1.0 - it / max(total_iters, 1))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history, best_miou, start_epoch = [], -1.0, 1

    if last.exists():                                           # resume an interrupted run
        ck = torch.load(last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["state_dict"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        history, best_miou, start_epoch = ck["history"], ck["best_miou"], ck["epoch"] + 1
        print(f"Resuming {method} from epoch {start_epoch}")
    else:
        info.update({"finetune_epochs": FT_EPOCHS, "parameters_encoder_head_total": count_parameters(model)})
        (root / "init_check.json").write_text(json.dumps(info, indent=2))
        print(f"{method}: start weights = {info['initialization']}"
              + (f" ({info.get('ssl_method')})" if method != "baseline" else ""))

    _generator.manual_seed(SEED + start_epoch)                  # identical batch order in all four runs

    for epoch in range(start_epoch, FT_EPOCHS + 1):
        start = time.time()
        model.train()
        loss_sum, seen = 0.0, 0

        for x, y in tqdm(train_loader, desc=f"{method} epoch {epoch}/{FT_EPOCHS}"):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(pixel_values=x).logits
            loss = seg_loss(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item() * x.size(0)
            seen += x.size(0)

        cms, _, val_loss, _ = evaluate_loader(model, val_loader)
        summary, _ = metrics_from_cm(cms.sum(axis=0))
        val_miou = summary["mIoU_7"]
        score = 0.0 if np.isnan(val_miou) else val_miou

        is_best = score > best_miou
        if is_best:
            best_miou = score
            torch.save({"state_dict": model.state_dict(), "epoch": epoch, "val_mIoU_7": val_miou,
                        "method": method}, best)

        history.append({
            "epoch": epoch,
            "train_loss": loss_sum / max(seen, 1),
            "val_loss": val_loss,
            "val_mIoU_7": val_miou,
            "val_pixel_accuracy": summary["pixel_accuracy"],
            "epoch_time_s": time.time() - start,
            "lr_encoder": optimizer.param_groups[0]["lr"],
            "lr_head": optimizer.param_groups[1]["lr"],
            "is_best": is_best,
        })
        pd.DataFrame(history).to_csv(root / "history.csv", index=False)
        torch.save({"state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": epoch, "history": history, "best_miou": best_miou}, last)
        print(f"{method} epoch {epoch}: train {history[-1]['train_loss']:.4f} | val {val_loss:.4f} | "
              f"val mIoU {val_miou:.4f} | {history[-1]['epoch_time_s']:.0f}s" + ("  <- best" if is_best else ""))
        if epoch < FT_EPOCHS and time_up(time.time() - start):
            break

    if not history or history[-1]["epoch"] < FT_EPOCHS:          # stopped by the time budget
        halt_notebook(f"fine-tuning '{method}'")
        del model, optimizer, scaler
        cleanup_gpu()
        return None

    assert best.exists(), f"Missing best.pt for {method}"
    done.write_text(json.dumps({"method": method, "epochs": FT_EPOCHS, "checkpoint": str(best),
                                "best_val_mIoU_7": best_miou}, indent=2))
    del model, optimizer, scaler
    cleanup_gpu()
    return str(best)


BEST_WEIGHTS = {}
print("Fine-tuning utilities ready:", {k: FT_CONFIG[k] for k in ["epochs", "batch", "optimizer", "baseline_init"]})
print("Runs:", EXPS)


In [ ]:
# CELL 22 — MODEL TRAINING: SegFormer baseline (random start weights, no SSL stage)
# Stage B: supervised fine-tuning for FT_EPOCHS epochs (one model per cell). Safe to re-run: finished runs are skipped.
print("Training BASELINE, epochs:", FT_EPOCHS)
BEST_WEIGHTS["baseline"] = train_experiment("baseline")
print("Saved best checkpoint:", BEST_WEIGHTS["baseline"])


In [ ]:
# CELL 23 — MODEL TRAINING: SegFormer + DINOv3 (encoder from dino/ssl_weights.pt)
# Stage B: supervised fine-tuning for FT_EPOCHS epochs (one model per cell). Safe to re-run: finished runs are skipped.
print("Training DINO, epochs:", FT_EPOCHS)
BEST_WEIGHTS["dino"] = train_experiment("dino")
print("Saved best checkpoint:", BEST_WEIGHTS["dino"])


In [ ]:
# CELL 24 — MODEL TRAINING: SegFormer + BYOL (encoder from byol/ssl_weights.pt)
# Stage B: supervised fine-tuning for FT_EPOCHS epochs (one model per cell). Safe to re-run: finished runs are skipped.
print("Training BYOL, epochs:", FT_EPOCHS)
BEST_WEIGHTS["byol"] = train_experiment("byol")
print("Saved best checkpoint:", BEST_WEIGHTS["byol"])


In [ ]:
# CELL 25 — MODEL TRAINING: SegFormer + MAE (encoder from mae/ssl_weights.pt)
# Stage B: supervised fine-tuning for FT_EPOCHS epochs (one model per cell). Safe to re-run: finished runs are skipped.
print("Training MAE, epochs:", FT_EPOCHS)
BEST_WEIGHTS["mae"] = train_experiment("mae")
print("Saved best checkpoint:", BEST_WEIGHTS["mae"])


In [ ]:
# ============================================================
# CELL 26 — CHECK TRAINING COMPLETION
# Purpose: Verify every run finished FT_EPOCHS epochs and show the epoch logs.
# ============================================================
def read_history(method):
    h = pd.read_csv(EXP_ROOTS[method] / "history.csv")
    return h.drop_duplicates("epoch", keep="last").sort_values("epoch").reset_index(drop=True)


HISTORIES = {m: read_history(m) for m in EXPS}

status_df = pd.DataFrame([{
    "model": m,
    "completed": (EXP_ROOTS[m] / "DONE.json").exists(),
    "best_exists": (EXP_ROOTS[m] / "best.pt").exists(),
    "ssl_weights_exist": m == "baseline" or (EXP_ROOTS[m] / "ssl_weights.pt").exists(),
    "epochs_recorded": int(HISTORIES[m]["epoch"].nunique()),
    "best_val_mIoU_7": HISTORIES[m]["val_mIoU_7"].max(),
    "best_epoch": int(HISTORIES[m].loc[HISTORIES[m]["val_mIoU_7"].idxmax(), "epoch"]),
} for m in EXPS]).set_index("model")
display(status_df.round(4))

assert status_df["completed"].all() and status_df["best_exists"].all() and status_df["ssl_weights_exist"].all()
assert (status_df["epochs_recorded"] == FT_EPOCHS).all()
BEST_WEIGHTS = {m: str(EXP_ROOTS[m] / "best.pt") for m in EXPS}
print("All four runs completed", FT_EPOCHS, "epochs each.")


In [ ]:
# ============================================================
# CELL 27 — MODEL-WISE TRAINING CURVES (helper + SegFormer baseline)
# Purpose: train/validation loss, validation mIoU, accuracy, epoch time for ONE model per cell.
# ============================================================
def plot_modelwise_curves(method):
    h = HISTORIES[method]
    panels = [
        ("Loss", [("train_loss", "Train loss"), ("val_loss", "Validation loss")]),
        ("Validation mIoU (7 classes)", [("val_mIoU_7", "Validation mIoU")]),
        ("Validation pixel accuracy", [("val_pixel_accuracy", "Validation accuracy")]),
        ("Epoch duration (s)", [("epoch_time_s", "Seconds per epoch")]),
    ]
    fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 4.8), squeeze=False)
    for ax, (title, series) in zip(axes.flat, panels):
        for col, label in series:
            ax.plot(h["epoch"], h[col], marker="o", linewidth=2, label=label)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel("Fine-tuning epoch")
        ax.set_xticks(EPOCH_TICKS)
        ax.grid(alpha=0.3)
        ax.legend(loc="best")
    fig.suptitle(f"{MODEL_LABELS_1LINE[method]} - Training Curves", fontsize=15, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG / f"{method}_modelwise_training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()


plot_modelwise_curves("baseline")


In [ ]:
# CELL 28 — MODEL-WISE TRAINING CURVES: SegFormer + DINOv3
plot_modelwise_curves("dino")


In [ ]:
# CELL 29 — MODEL-WISE TRAINING CURVES: SegFormer + BYOL
plot_modelwise_curves("byol")


In [ ]:
# CELL 30 — MODEL-WISE TRAINING CURVES: SegFormer + MAE
plot_modelwise_curves("mae")


In [ ]:
# ============================================================
# CELL 31 — VALIDATION SET EVALUATION (all four runs)
# Purpose: Score each best.pt on the validation set (used for monitoring only; never for test tuning).
# ============================================================
VAL_CMS, val_rows, val_pc = {}, {}, {}
for method in EXPS:
    model = load_best_model(method)
    cms_val, uids_val, _, val_empty = evaluate_loader(
        model, val_loader, save_dir=EV / "predictions" / "val" / method)
    np.savez_compressed(EV / f"val_{method}.npz", cms=cms_val, uids=np.asarray(uids_val))
    metrics, per_class = metrics_from_cm(cms_val.sum(axis=0))
    metrics["no_predictions"] = val_empty
    VAL_CMS[method], val_rows[method], val_pc[method] = cms_val, metrics, per_class
    del model
    cleanup_gpu()

val_summary = pd.DataFrame.from_dict(val_rows, orient="index").loc[EXPS]
val_summary.to_csv(EV / "val_summary.csv")
for metric in ["IoU", "Dice", "Precision", "Recall", "F1"]:
    pd.DataFrame({m: val_pc[m][metric] for m in EXPS}).to_csv(EV / f"val_per_class_{metric.lower()}.csv")

display(val_summary.round(4))


In [ ]:
# ============================================================
# CELL 32 — FINAL TEST EVALUATION (single scoring run per model)
# Purpose: Score the held-out test set ONCE per run with best.pt. If the result already exists on
#          disk it is reloaded, never recomputed. Nothing is tuned on the test set.
# ============================================================
TEST_CMS, TEST_UIDS, test_rows, TEST_PER_CLASS = {}, {}, {}, {}
for method in EXPS:
    cache = EV / f"test_{method}.npz"
    pred_dir = EV / "predictions" / "test" / method
    if cache.exists():
        saved = np.load(cache)
        cms_test, uids_test = saved["cms"], [str(u) for u in saved["uids"]]
        test_empty = int(sum(not np.asarray(Image.open(p)).any() for p in pred_dir.glob("*.png")))
        print(f"{method}: test set already scored once - reloaded.")
    else:
        model = load_best_model(method)
        cms_test, uids_test, _, test_empty = evaluate_loader(model, test_loader, save_dir=pred_dir)
        np.savez_compressed(cache, cms=cms_test, uids=np.asarray(uids_test))
        del model
        cleanup_gpu()
    metrics, per_class = metrics_from_cm(cms_test.sum(axis=0))
    metrics["no_predictions"] = test_empty
    TEST_CMS[method], TEST_UIDS[method], test_rows[method], TEST_PER_CLASS[method] = cms_test, uids_test, metrics, per_class

assert all(TEST_UIDS[m] == TEST_UIDS["baseline"] for m in EXPS), "test image order differs between runs"

test_summary = pd.DataFrame.from_dict(test_rows, orient="index").loc[EXPS]
test_summary.to_csv(EV / "test_summary.csv")
for metric in ["IoU", "Dice", "Precision", "Recall", "F1"]:
    pd.DataFrame({m: TEST_PER_CLASS[m][metric] for m in EXPS}).to_csv(EV / f"test_per_class_{metric.lower()}.csv")

print("FINAL TEST RESULTS:")
display(test_summary.round(4))


In [ ]:
# ============================================================
# CELL 33 — CONSOLIDATED EVALUATION METRICS
# Purpose: Display all evaluation metrics in one cell.
# ============================================================
print("========== VALIDATION METRICS ==========")
display(pd.read_csv(EV / "val_summary.csv", index_col=0).round(4))

print("========== TEST METRICS ==========")
display(pd.read_csv(EV / "test_summary.csv", index_col=0).round(4))

for metric in ["iou", "dice", "precision", "recall", "f1"]:
    print(f"PER-CLASS {metric.upper()} (test)")
    display(pd.read_csv(EV / f"test_per_class_{metric}.csv", index_col=0).round(4))


In [ ]:
# ============================================================
# CELL 34 — PAIRWISE COMPARISON: helper function (NO plots here)
# Purpose: Compare two runs: supervised curves (train loss, validation loss, validation mIoU) and
#          the test-metric differences. Each pair is drawn in its OWN following cell.
# ============================================================
def pairwise_plots(method_a, method_b):
    la, lb = DISPLAY_NAMES[method_a], DISPLAY_NAMES[method_b]
    keys = [("train_loss", "Train loss"), ("val_loss", "Validation loss"), ("val_mIoU_7", "Validation mIoU (7 classes)")]
    fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4.6), squeeze=False)
    for ax, (key, title) in zip(axes.flat, keys):
        for m, label in ((method_a, la), (method_b, lb)):
            ax.plot(HISTORIES[m]["epoch"], HISTORIES[m][key], marker="o", linewidth=2,
                    color=MODEL_COLORS[m], label=label)
        ax.set(title=title, xlabel="Fine-tuning epoch")
        ax.set_xticks(EPOCH_TICKS)
        ax.grid(alpha=.3)
        ax.legend(loc="best")
    fig.suptitle(f"{la} vs {lb}", fontsize=15, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG / f"{method_a}_vs_{method_b}_loss_miou.png", dpi=150, bbox_inches="tight")
    plt.show()

    summary = pd.read_csv(EV / "test_summary.csv", index_col=0)
    numeric = summary.loc[[method_a, method_b]].select_dtypes(include=[np.number])
    delta = numeric.loc[method_b] - numeric.loc[method_a]
    diff = pd.DataFrame({la: numeric.loc[method_a], lb: numeric.loc[method_b],
                         f"Delta ({lb} minus {la})": delta})
    display(diff.round(4))
    diff.to_csv(EV / f"{method_a}_vs_{method_b}_metric_differences.csv")

    preferred = [c for c in ["mIoU_7", "mDice_7", "mF1_fg6", "pixel_accuracy"] if c in numeric.columns]
    fig, ax = plt.subplots(figsize=(8, 4))
    delta.loc[preferred].plot(kind="bar", ax=ax, color="#0072B2", edgecolor="black")
    ax.axhline(0, color="black", lw=.8)
    ax.set(title=f"Test metric difference: {lb} - {la}", ylabel="Absolute metric difference")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=.3)
    fig.tight_layout()
    fig.savefig(FIG / f"{method_a}_vs_{method_b}_metric_delta.png", dpi=150)
    plt.show()


print("pairwise_plots(method_a, method_b) is ready.")


In [ ]:
# CELL 35 — PAIRWISE: SegFormer baseline vs SegFormer + DINOv3
pairwise_plots("baseline", "dino")


In [ ]:
# CELL 36 — PAIRWISE: SegFormer baseline vs SegFormer + BYOL
pairwise_plots("baseline", "byol")


In [ ]:
# CELL 37 — PAIRWISE: SegFormer baseline vs SegFormer + MAE
pairwise_plots("baseline", "mae")


In [ ]:
# CELL 38 — ALL FOUR RUNS TOGETHER: validation mIoU and train loss across fine-tuning epochs
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, (key, title) in zip(axes, [("train_loss", "Train loss"), ("val_loss", "Validation loss"),
                                   ("val_mIoU_7", "Validation mIoU (7 classes)")]):
    for m in EXPS:
        ax.plot(HISTORIES[m]["epoch"], HISTORIES[m][key], marker="o", linewidth=2,
                color=MODEL_COLORS[m], label=DISPLAY_NAMES[m])
    ax.set(title=title, xlabel="Fine-tuning epoch")
    ax.set_xticks(EPOCH_TICKS)
    ax.grid(alpha=.3)
    ax.legend()
fig.suptitle("SegFormer - Baseline vs DINOv3 vs BYOL vs MAE", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "all_models_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# CELL 39 — CONFUSION MATRIX, compact "classic" style (green-yellow, Background removed)
# Same look as the YOLO26s notebook: 6 disease classes only, count + row % in every cell,
# number colour chosen from the cell brightness.
# Title: "<CM_TITLE_NAME> Confusion Matrix (<approach>)".
# ============================================================
from matplotlib.colors import LinearSegmentedColormap

CM_COLOR_BY = "count"                  # "count" or "percent"
CM_TITLE_NAME = "SegFormer"
CM_MODEL_TAG = {"baseline": "baseline", "dino": "DINOv3", "byol": "BYOL", "mae": "MAE"}
CM_CMAP = LinearSegmentedColormap.from_list(
    "cm_green_yellow", ["#006400", "#0F8A18", "#66DD22", "#FFFF00"])


def _count_text(n):
    n = int(n)
    if n >= 1_000_000:
        return f"{n / 1e6:.1f}M"
    if n >= 10_000:
        return f"{n / 1e3:.0f}K"
    return f"{n:,}"


def plot_confusion_matrix(method="baseline", split="test", show=True):
    saved = np.load(EV / f"{split}_{method}.npz")
    cm_full = saved["cms"].sum(axis=0).astype(float)
    cm = cm_full[1:, 1:]                                   # 6 disease classes (Background dropped)
    n = cm.shape[0]
    pct = 100 * cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    names = [MASK_NAMES[i].replace("_", " ").upper() for i in range(1, n + 1)]

    data = cm if CM_COLOR_BY == "count" else pct
    vmax = max(data.max(), 1)

    fig, ax = plt.subplots(figsize=(8.5, 7))
    ax.imshow(data, cmap=CM_CMAP, vmin=0, vmax=vmax, aspect="auto")
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=11)
    ax.set_yticklabels(names, fontsize=11)
    ax.set_xlabel("Predicted", fontsize=12, labelpad=10)
    ax.set_ylabel("True", fontsize=12, labelpad=10)
    ax.set_xticks(np.arange(-.5, n, 1), minor=True)
    ax.set_yticks(np.arange(-.5, n, 1), minor=True)
    ax.grid(which="minor", color="black", linewidth=2)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", length=0)

    for i in range(n):
        for j in range(n):
            dark_text = (data[i, j] / vmax) > 0.22
            ax.text(j, i, f"{_count_text(cm[i, j])}\n{pct[i, j]:.1f}%",
                    ha="center", va="center", fontsize=13, fontweight="bold",
                    color="black" if dark_text else "white")

    ax.set_title(f"{CM_TITLE_NAME} Confusion Matrix ({CM_MODEL_TAG.get(method, method)})",
                 fontsize=15, fontweight="bold", pad=10)
    fig.tight_layout()
    out = FIG / f"{method}_confusion_{split}.png"
    fig.savefig(out, dpi=200)
    if show:
        plt.show()
    plt.close(fig)
    return out


print("plot_confusion_matrix(method) is ready.")


In [ ]:
# CELL 40 — Confusion matrix: SegFormer (baseline) — test set, pixel counts and row-normalised %
plot_confusion_matrix("baseline")


In [ ]:
# CELL 41 — Confusion matrix: SegFormer (DINOv3) — test set, pixel counts and row-normalised %
plot_confusion_matrix("dino")


In [ ]:
# CELL 42 — Confusion matrix: SegFormer (BYOL) — test set, pixel counts and row-normalised %
plot_confusion_matrix("byol")


In [ ]:
# CELL 43 — Confusion matrix: SegFormer (MAE) — test set, pixel counts and row-normalised %
plot_confusion_matrix("mae")


In [ ]:
# ============================================================
# CELL 44 — QUALITATIVE SEGMENTATION (one test image per class, all four models)
# Purpose: Show input, ground truth and the prediction of every model side by side.
# ============================================================
test_table = man[man["split"] == "test"].sort_values("uid")
selected_uids = [str(test_table[test_table["class_name"] == c].iloc[0]["uid"]) for c in CLASS_NAMES]

titles = ["Original", "Ground truth"] + [MODEL_LABELS_1LINE[m] for m in EXPS]
fig, axes = plt.subplots(len(selected_uids), len(titles), figsize=(3.6 * len(titles), 3.4 * len(selected_uids)),
                         squeeze=False)
for row, uid in enumerate(selected_uids):
    image = load_rgb(IMAGE_INDEX["test"][uid])
    gt = load_gt("test", uid)
    panels = [image, color_mask(gt)] + [
        color_mask(np.asarray(Image.open(EV / "predictions/test" / m / f"{uid}.png"))) for m in EXPS]
    for col, panel in enumerate(panels):
        axes[row, col].imshow(panel)
        axes[row, col].axis("off")
        axes[row, col].grid(False)
        if row == 0:
            axes[row, col].set_title(titles[col], fontsize=12)
fig.tight_layout()
fig.savefig(FIG / "all_models_qualitative.png", dpi=130)
plt.show()


In [ ]:
# ============================================================
# CELL 45 — THREE BEST AND THREE WORST TEST IMAGES per model (IoU of the image's true class)
# Purpose: "Show Results" requirement. No selection or tuning uses this.
# ============================================================
label_of = dict(zip(man["uid"], man["label"].astype(int)))


def best_worst_figure(method):
    rows = []
    for cm, uid in zip(TEST_CMS[method], TEST_UIDS[method]):
        c = label_of[uid]
        union = cm[c, :].sum() + cm[:, c].sum() - cm[c, c]
        rows.append({"uid": uid, "class_name": MASK_NAMES[c], "IoU_true_class": cm[c, c] / union if union else 0.0})
    rank_df = pd.DataFrame(rows).sort_values("IoU_true_class", ascending=False).reset_index(drop=True)
    rank_df.to_csv(EV / f"test_image_iou_ranking_{method}.csv", index=False)

    picks = [("Best", r) for r in rank_df.head(3).itertuples()] + \
            [("Worst", r) for r in rank_df.tail(3).iloc[::-1].itertuples()]
    fig, axes = plt.subplots(len(picks), 4, figsize=(16, 3.6 * len(picks)), squeeze=False)
    for row_i, (tag, r) in enumerate(picks):
        rgb = load_rgb(IMAGE_INDEX["test"][r.uid])
        gt = load_gt("test", r.uid)
        pred = np.asarray(Image.open(EV / "predictions/test" / method / f"{r.uid}.png"))
        overlay = np.where((pred > 0)[..., None], (rgb * 0.5 + color_mask(pred) * 0.5).astype(np.uint8), rgb)
        for col_i, (panel, title) in enumerate(zip(
                [rgb, color_mask(gt), color_mask(pred), overlay],
                [f"{tag} - {r.class_name}", "Ground truth", f"Prediction (IoU {r.IoU_true_class:.2f})", "Overlay"])):
            axes[row_i, col_i].imshow(panel)
            axes[row_i, col_i].set_title(title, fontsize=13)
            axes[row_i, col_i].axis("off"); axes[row_i, col_i].grid(False)
    fig.suptitle(f"{MODEL_LABELS_1LINE[method]} - Three Best and Three Worst Test Images", fontsize=15,
                 fontweight="bold", y=1.0)
    fig.tight_layout()
    fig.savefig(FIG / f"{method}_best_worst.png", dpi=110)
    plt.show()


for method in EXPS:
    best_worst_figure(method)


In [ ]:
# ============================================================
# CELL 46 — GRAD-CAM FOR SEGFORMER (class-score Grad-CAM, all four models)
# Purpose: Show which image regions drive the score of one class.
# Layer   : last MiT encoder stage (GRADCAM_STAGE = -1; use 2 for a finer 1/16-scale map).
# Score   : mean logit of the target class over the pixels the model assigns to that class;
#           if it assigns none (common after 2 epochs) the ground-truth region of the class is used.
# The map is drawn on the SEMANTIC MASK, not on the original photo.
# ============================================================
GRADCAM_STAGE = -1
TEST_FILES = [IMAGE_INDEX["test"][k] for k in sorted(IMAGE_INDEX["test"])]
_cam_cmap = plt.get_cmap("turbo")


def gradcam_segformer(model, rgb, class_id, gt512=None):
    model.eval()
    x = preprocess_rgb(rgb).unsqueeze(0).to(DEVICE)

    with torch.enable_grad():
        model.zero_grad(set_to_none=True)
        out = model(pixel_values=x, output_hidden_states=True)
        feats = out.hidden_states[GRADCAM_STAGE]
        feats.retain_grad()
        up = F.interpolate(out.logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

        region = up[0].argmax(0) == class_id
        if int(region.sum()) == 0 and gt512 is not None:
            region = torch.from_numpy(gt512 == class_id).to(up.device)
        if int(region.sum()) == 0:
            region = torch.ones_like(region, dtype=torch.bool)

        score = up[0, class_id][region].mean()
        score.backward()

    weights = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * feats).sum(dim=1, keepdim=True)).detach()
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    model.zero_grad(set_to_none=True)
    return cam.cpu().numpy(), tuple(feats.shape), bool(feats.requires_grad)


def cam_overlay(base_rgb, cam):
    rgba = _cam_cmap(cam)
    alpha = (cam ** 0.8 * 0.8)[..., None]
    blended = base_rgb.astype(np.float32) / 255 * (1 - alpha) + rgba[..., :3] * alpha
    return (np.clip(blended, 0, 1) * 255).astype(np.uint8)


def main_class(gt):
    present, counts = np.unique(gt[gt > 0], return_counts=True)
    return int(present[np.argmax(counts)]) if len(present) else 1

sample_path = TEST_FILES[0]
sample_rgb = load_rgb(sample_path)
sample_gt = load_gt("test", sample_path.stem)
TARGET_CLASS = main_class(sample_gt)                           # mask value 1..6
gt512 = cv2.resize(sample_gt, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
mask_rgb512 = color_mask(gt512)
print("Image:", sample_path.name, "| target class:", MASK_NAMES[TARGET_CLASS])

fig, axes = plt.subplots(1, 1 + len(EXPS), figsize=(5.2 * (1 + len(EXPS)), 5.6))
axes[0].imshow(mask_rgb512); axes[0].set_title("Semantic Mask")
for ax, method in zip(axes[1:], EXPS):
    cam_model = load_best_model(method)
    cam, act_shape, tracked = gradcam_segformer(cam_model, sample_rgb, TARGET_CLASS, gt512)
    print(f"{method}: activation {act_shape}, gradient tracking {tracked}")
    ax.imshow(cam_overlay(mask_rgb512, cam))
    ax.set_title(f"{MODEL_LABELS_1LINE[method]}\nGrad-CAM", fontsize=13)
    del cam_model
    cleanup_gpu()
for ax in axes:
    ax.axis("off"); ax.grid(False)
fig.suptitle(f"SegFormer Grad-CAM - {MASK_NAMES[TARGET_CLASS]}", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "gradcam_all_models_sample.png", dpi=140)
plt.show()


In [ ]:
# ============================================================
# CELL 47 — LIME EXPLAINABILITY (all four models)
# Purpose: Explain which superpixels raise the model's class probability.
# SegFormer gives real per-pixel probabilities, so LIME explains the image-average
# softmax probability of one class (this also works when argmax predicts only background).
# Explained class = the class (1..6) with the highest average probability on the image.
# ============================================================
from lime import lime_image
from skimage.segmentation import slic, mark_boundaries

LIME_SIZE = (640, 480)                                          # (width, height)
lime_model = None


def use_lime_model(method):
    """Select which run's best.pt LIME (and the Gradio app) explains."""
    global lime_model
    lime_model = load_best_model(method)
    return lime_model


@torch.no_grad()
def lime_predict(images):
    outputs = []
    for start in range(0, len(images), 16):
        batch = torch.stack([preprocess_rgb(np.asarray(im, dtype=np.uint8))
                             for im in images[start:start + 16]]).to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = lime_model(pixel_values=batch).logits
        outputs.append(logits.float().softmax(dim=1).mean(dim=(2, 3)).cpu().numpy())
    return np.concatenate(outputs)


def explain_lime_rgb(rgb, num_samples=200, class_id=None):
    """LIME on one RGB image (already resized to LIME_SIZE). Returns (explanation, class id)."""
    coverage = lime_predict([rgb])[0]
    if class_id is None:
        class_id = int(np.argmax(coverage[1:])) + 1
    explanation = lime_image.LimeImageExplainer(random_state=SEED).explain_instance(
        rgb, lime_predict, labels=(class_id,), top_labels=None,
        num_samples=num_samples, batch_size=16, hide_color=0,
        segmentation_fn=lambda image: slic(image, n_segments=55, compactness=10, sigma=1, start_label=0))
    return explanation, class_id


def explain_lime(uid, num_samples=200, class_id=None):
    rgb = cv2.resize(load_rgb(IMAGE_INDEX["test"][uid]), LIME_SIZE, interpolation=cv2.INTER_AREA)
    return explain_lime_rgb(rgb, num_samples, class_id)


def lime_panels(uid, num_samples=200, axes=None):
    """Draw: semantic mask | top supporting regions | superpixel weights. Returns the class id."""
    explanation, class_id = explain_lime(uid, num_samples)
    gt = cv2.resize(load_gt("test", uid), LIME_SIZE, interpolation=cv2.INTER_NEAREST)
    mask_rgb = color_mask(gt)

    _, regions = explanation.get_image_and_mask(class_id, positive_only=True, num_features=8, hide_rest=False)
    tint = mask_rgb.astype(np.float32) / 255
    tint[regions > 0] = tint[regions > 0] * 0.55 + np.array([0.0, 0.4, 0.0]) * 0.45
    top = mark_boundaries(tint, regions, color=(1, 0.9, 0), mode="thick")

    weight_map = np.zeros(explanation.segments.shape, dtype=np.float32)
    for seg_id, weight in explanation.local_exp[class_id]:
        weight_map[explanation.segments == seg_id] = weight
    limit = max(float(np.abs(weight_map).max()), 1e-8)

    if axes is None:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
    axes[0].imshow(mask_rgb); axes[0].set_title(f"Semantic mask - {MASK_NAMES[int(main_class(gt))]}", fontsize=12)
    axes[1].imshow(top); axes[1].set_title(f"LIME: top supporting regions\nexplained class = {MASK_NAMES[class_id]}", fontsize=12)
    im = axes[2].imshow(weight_map, cmap="coolwarm", vmin=-limit, vmax=limit)
    axes[2].set_title("Superpixel weights\n(red = supports class, blue = opposes)", fontsize=12)
    for ax in axes:
        ax.axis("off"); ax.grid(False)
    return class_id, im

lime_uid = TEST_FILES[0].stem
print("LIME image:", TEST_FILES[0].name)
fig, axes = plt.subplots(len(EXPS), 3, figsize=(18, 5.2 * len(EXPS)))
for row, method in enumerate(EXPS):
    use_lime_model(method)
    LIME_CLASS, _im = lime_panels(lime_uid, num_samples=200, axes=axes[row])
    axes[row, 0].set_ylabel(MODEL_LABELS_1LINE[method], fontsize=14, fontweight="bold")
    axes[row, 0].axis("on")
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    fig.colorbar(_im, ax=axes[row, 2], fraction=0.046, label="LIME weight")
    print(f"{method}: explained class = {MASK_NAMES[LIME_CLASS]}")
fig.suptitle("LIME - SegFormer (all four models)", fontsize=16, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / f"lime_all_models_e{FT_EPOCHS}.png", dpi=120)
plt.show()


In [ ]:
# ============================================================
# CELL 48 — COMPUTATIONAL PERFORMANCE ANALYSIS (all four models)
# Purpose: Parameters, model size, GMACs, latency, FPS and GPU memory.
# GMACs are counted at the training input size (512x512) with torch FlopCounterMode (FLOPs / 2).
# Latency = pre-processing + network + up-sampling/argmax for ONE image (GPU, after warm-up).
# ============================================================
from torch.utils.flop_counter import FlopCounterMode


def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


sample_rgb = load_rgb(TEST_FILES[0])
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
compute_rows = []

for method in EXPS:
    prof_model = load_best_model(method)
    num_parameters = sum(p.numel() for p in prof_model.parameters())
    model_size_mb = (EXP_ROOTS[method] / "best.pt").stat().st_size / (1024 ** 2)

    try:
        with torch.no_grad(), FlopCounterMode(display=False) as flop_counter:
            prof_model(pixel_values=dummy)
        gmacs = flop_counter.get_total_flops() / 2 / 1e9
    except Exception as error:
        print("MAC profiling issue:", error)
        gmacs = float("nan")

    for _ in range(3):                                          # warm-up
        predict_image(prof_model, sample_rgb)
    sync()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    pipeline_ms, network_ms = [], []
    for _ in range(15):
        sync(); t0 = time.perf_counter()
        predict_image(prof_model, sample_rgb)
        sync(); pipeline_ms.append((time.perf_counter() - t0) * 1000)

        sync(); t0 = time.perf_counter()
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            prof_model(pixel_values=dummy)
        sync(); network_ms.append((time.perf_counter() - t0) * 1000)

    avg_latency = float(np.mean(pipeline_ms))
    peak_memory = (torch.cuda.max_memory_allocated() / (1024 ** 2)) if torch.cuda.is_available() else float("nan")
    compute_rows.append({
        "model": method,
        "parameters_M": num_parameters / 1e6,
        "model_size_MB": model_size_mb,
        "estimated_GMACs": gmacs,
        "inference_ms": float(np.mean(network_ms)),
        "latency_ms": avg_latency,
        "FPS": 1000 / avg_latency if avg_latency > 0 else float("nan"),
        "peak_inference_GPU_MB": peak_memory,
    })
    del prof_model
    cleanup_gpu()

compute_df = pd.DataFrame(compute_rows).set_index("model")
compute_df.to_csv(EV / "compute_profile.csv")
display(compute_df.round(3))


In [ ]:
# ============================================================
# CELL 49 — FINAL INFERENCE PIPELINE (all four models)
# Purpose: Show input image and the predicted overlay of every model for a few test images.
# ============================================================
demo_uids = [TEST_FILES[i].stem for i in np.linspace(0, len(TEST_FILES) - 1, 3).astype(int)]
demo_rgb = {uid: load_rgb(IMAGE_INDEX["test"][uid]) for uid in demo_uids}
demo_pred = {m: {} for m in EXPS}
for method in EXPS:
    infer_model = load_best_model(method)
    for uid in demo_uids:
        demo_pred[method][uid] = predict_image(infer_model, demo_rgb[uid])
    del infer_model
    cleanup_gpu()

fig, axes = plt.subplots(len(demo_uids), 1 + len(EXPS), figsize=(4.2 * (1 + len(EXPS)), 4.0 * len(demo_uids)),
                         squeeze=False)
for row, uid in enumerate(demo_uids):
    rgb = demo_rgb[uid]
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title("Input image" if row == 0 else "")
    for col, method in enumerate(EXPS, start=1):
        pred = demo_pred[method][uid]
        overlay = np.where((pred > 0)[..., None], (rgb * 0.5 + color_mask(pred) * 0.5).astype(np.uint8), rgb)
        axes[row, col].imshow(overlay)
        axes[row, col].set_title(MODEL_LABELS_1LINE[method] if row == 0 else "", fontsize=12)
    for ax in axes[row]:
        ax.axis("off"); ax.grid(False)
fig.suptitle("SegFormer - Inference Pipeline (mask overlay)", fontsize=15, fontweight="bold", y=1.0)
fig.tight_layout()
fig.savefig(FIG / "all_models_inference_pipeline.png", dpi=120)
plt.show()
print("Inference completed.")


In [ ]:
# ============================================================
# CELL 50 — MCC, MACRO-F1, 1,000-RESAMPLE BOOTSTRAP, PAIRED DIFFERENCES, test_metrics.json
# Purpose: Multiclass MCC, Macro-F1 and image-level bootstrap 95% CIs for mIoU / Macro-F1 / MCC for
#          all four runs, per-class stability, and PAIRED bootstrap differences (each SSL method minus
#          the baseline, same resampled test images) with the verdict BETTER / SAME / WORSE:
#          BETTER = whole 95% CI above 0, WORSE = whole CI below 0, otherwise SAME. Verdict uses mIoU.
# Saves: eval/test_bootstrap_1000.csv, eval/test_metrics.json, <method>/test_metrics.json
# ============================================================
N_BOOT = 1000
SCORE_KEYS = ["mIoU_7", "MacroF1_7", "MCC"]


def mcc_from_cm(cm):
    cm = cm.astype(np.float64)
    s, c = cm.sum(), np.trace(cm)
    t, p = cm.sum(axis=1), cm.sum(axis=0)
    denom = np.sqrt((s ** 2 - (p ** 2).sum()) * (s ** 2 - (t ** 2).sum()))
    return float((c * s - (p * t).sum()) / denom) if denom > 0 else 0.0


def scores_from_cm(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = np.where(tp + fp + fn > 0, tp / (tp + fp + fn), np.nan)
        f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan)
    return dict(mIoU_7=np.nanmean(iou), MacroF1_7=np.nanmean(f1), MCC=mcc_from_cm(cm), iou_per_class=iou)


n_img = len(TEST_CMS["baseline"])
rng = np.random.default_rng(SEED)
boot_idx = rng.integers(0, n_img, size=(N_BOOT, n_img))           # the SAME resamples for every run (paired)

boot = {m: {"mIoU_7": [], "MacroF1_7": [], "MCC": [], "iou_pc": []} for m in EXPS}
for idx in tqdm(boot_idx, desc="bootstrap"):
    for m in EXPS:
        sc = scores_from_cm(TEST_CMS[m][idx].sum(axis=0))
        for k in SCORE_KEYS:
            boot[m][k].append(sc[k])
        boot[m]["iou_pc"].append(sc["iou_per_class"])
boot = {m: {k: np.asarray(v) for k, v in d.items()} for m, d in boot.items()}


def verdict_from_ci(lo, hi):
    return "BETTER" if lo > 0 else ("WORSE" if hi < 0 else "SAME")


TEST_METRICS, entries = {}, []
for m in EXPS:
    point = scores_from_cm(TEST_CMS[m].sum(axis=0))
    entry = {"model": m}
    for k in SCORE_KEYS:
        lo, hi = np.percentile(boot[m][k], [2.5, 97.5])
        entry.update({k: float(point[k]), f"{k}_CI_low": float(lo), f"{k}_CI_high": float(hi)})
        if m == "baseline":
            d_point, d_lo, d_hi = 0.0, 0.0, 0.0
        else:
            diff = boot[m][k] - boot["baseline"][k]
            d_lo, d_hi = np.percentile(diff, [2.5, 97.5])
            d_point = float(point[k] - scores_from_cm(TEST_CMS["baseline"].sum(axis=0))[k])
        entry.update({f"{k}_diff": float(d_point), f"{k}_diff_low": float(d_lo), f"{k}_diff_high": float(d_hi)})
        entry[f"verdict_{k}"] = "reference" if m == "baseline" else verdict_from_ci(d_lo, d_hi)
    entry["verdict_vs_baseline"] = entry["verdict_mIoU_7"]
    entry.update({k: float(v) for k, v in test_summary.loc[m].items()})
    entries.append(entry)
    TEST_METRICS[m] = entry
    (EXP_ROOTS[m] / "test_metrics.json").write_text(json.dumps({m: entry}, indent=2))   # one JSON per run

boot_df = pd.DataFrame(entries).set_index("model")
boot_df.to_csv(EV / "test_bootstrap_1000.csv")
(EV / "test_metrics.json").write_text(json.dumps(TEST_METRICS, indent=2))

display(boot_df[["mIoU_7", "mIoU_7_CI_low", "mIoU_7_CI_high", "MacroF1_7", "MacroF1_7_CI_low",
                 "MacroF1_7_CI_high", "MCC", "MCC_CI_low", "MCC_CI_high"]].round(4))
print("PAIRED differences vs baseline (SSL minus baseline, 95% CI):")
display(boot_df.loc[[m for m in EXPS if m != "baseline"],
                    ["mIoU_7_diff", "mIoU_7_diff_low", "mIoU_7_diff_high", "MacroF1_7_diff", "MacroF1_7_diff_low",
                     "MacroF1_7_diff_high", "MCC_diff", "MCC_diff_low", "MCC_diff_high",
                     "verdict_mIoU_7", "verdict_MacroF1_7", "verdict_MCC"]].round(4))

# ---- per-class stability (mean +- std of IoU over the bootstrap resamples) ----
stability = {}
for m in EXPS:
    mu, sd = np.nanmean(boot[m]["iou_pc"], axis=0), np.nanstd(boot[m]["iou_pc"], axis=0)
    stability[m] = [f"{a:.3f} ± {b:.3f}" for a, b in zip(mu, sd)]
stability_df = pd.DataFrame(stability, index=[MASK_NAMES[i] for i in range(NUM_CLASSES)])
stability_df.to_csv(EV / "per_class_stability.csv")
print("Per-class IoU stability (mean ± std over 1,000 bootstrap resamples):")
display(stability_df)

# ---- 95% CI plot (all four runs) ----
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, k in zip(axes, SCORE_KEYS):
    for i, m in enumerate(EXPS):
        e = TEST_METRICS[m]
        ax.errorbar(e[k], i, xerr=[[e[k] - e[f"{k}_CI_low"]], [e[f"{k}_CI_high"] - e[k]]], fmt="o", ms=11,
                    capsize=7, elinewidth=3, color=MODEL_COLORS[m])
        ax.text(e[f"{k}_CI_high"], i + 0.12, f" {e[k]:.3f}", fontsize=12, fontweight="bold", va="bottom")
    ax.set_yticks(range(len(EXPS)))
    ax.set_yticklabels([DISPLAY_NAMES[m] for m in EXPS])
    ax.invert_yaxis(); ax.margins(x=0.2)
    ax.set_title(f"{k} (95% CI)")
fig.suptitle("SegFormer - 1,000-Resample Bootstrap Confidence Intervals", fontsize=15, fontweight="bold", y=1.03)
fig.tight_layout()
fig.savefig(FIG / "bootstrap_ci_all_models.png")
plt.show()

# ---- paired differences plot ----
fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))
for ax, k in zip(axes, SCORE_KEYS):
    for i, m in enumerate([x for x in EXPS if x != "baseline"]):
        e = TEST_METRICS[m]
        ax.errorbar(e[f"{k}_diff"], i, xerr=[[e[f"{k}_diff"] - e[f"{k}_diff_low"]], [e[f"{k}_diff_high"] - e[f"{k}_diff"]]],
                    fmt="o", ms=11, capsize=7, elinewidth=3, color=MODEL_COLORS[m])
        ax.text(e[f"{k}_diff_high"], i + 0.12, f" {e['verdict_' + k]}", fontsize=12, fontweight="bold", va="bottom")
    ax.axvline(0, color="black", lw=1.2, ls="--")
    ax.set_yticks(range(3)); ax.set_yticklabels([DISPLAY_NAMES[x] for x in EXPS if x != "baseline"])
    ax.invert_yaxis(); ax.margins(x=0.3)
    ax.set_title(f"{k}: SSL minus baseline")
fig.suptitle("Paired bootstrap differences vs baseline (95% CI)", fontsize=15, fontweight="bold", y=1.04)
fig.tight_layout()
fig.savefig(FIG / "paired_bootstrap_vs_baseline.png")
plt.show()
print("Saved:", EV / "test_metrics.json", "and one test_metrics.json inside each run folder")


In [ ]:
# ============================================================
# CELL 51 — METRIC GRAPHS: mIoU, Accuracy, Precision, Recall, F1 (all four runs)
# Purpose: One grouped bar chart of the headline test metrics, values printed on the bars.
# Precision is averaged only over classes the model actually predicted (undefined otherwise).
# ============================================================
metric_defs = [("mIoU (7 classes)", "mIoU_7"), ("Pixel accuracy", "pixel_accuracy"),
               ("Precision (fg, macro)", "mPrecision_fg6"), ("Recall (fg, macro)", "mRecall_fg6"),
               ("F1 (fg, macro)", "mF1_fg6")]
fig, ax = plt.subplots(figsize=(17, 6.5))
width = 0.2
for j, m in enumerate(EXPS):
    for i, (label, col) in enumerate(metric_defs):
        value = test_summary.loc[m, col]
        shown = 0.0 if np.isnan(value) else value
        x = i + (j - 1.5) * width
        ax.bar(x, shown, width, color=MODEL_COLORS[m], edgecolor="black", label=DISPLAY_NAMES[m] if i == 0 else None)
        ax.text(x, shown + 0.012, "n/a" if np.isnan(value) else f"{value:.2f}", ha="center", fontsize=10, rotation=90)
ax.set_xticks(range(len(metric_defs)))
ax.set_xticklabels([d[0] for d in metric_defs], rotation=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.legend(ncol=4, loc="upper right")
ax.set_title("SegFormer - Test Metrics (baseline vs DINOv3 vs BYOL vs MAE)", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "all_models_metric_bars.png")
plt.show()


In [ ]:
# ============================================================
# CELL 52 — PER-CLASS IoU AND F1 (grouped bars, all four runs)
# Purpose: Show which disease classes each model handles best.
# "n/a" = undefined (the model never predicted that class).
# ============================================================
for metric in ["IoU", "F1"]:
    table = pd.DataFrame({m: TEST_PER_CLASS[m][metric] for m in EXPS})
    fig, ax = plt.subplots(figsize=(17, 6))
    width = 0.2
    for j, m in enumerate(EXPS):
        for c_i, value in enumerate(table[m].values):
            x = c_i + (j - 1.5) * width
            ax.bar(x, 0 if np.isnan(value) else value, width, color=MODEL_COLORS[m], edgecolor="black",
                   label=DISPLAY_NAMES[m] if c_i == 0 else None)
            if np.isnan(value):
                ax.text(x, 0.01, "n/a", ha="center", fontsize=9, rotation=90)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels([MASK_NAMES[i].replace("_", " ").title() for i in range(NUM_CLASSES)], rotation=15)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(metric)
    ax.legend(ncol=4, loc="upper right")
    ax.set_title(f"SegFormer - Per-Class Test {metric}", fontsize=15, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG / f"all_models_per_class_{metric.lower()}.png")
    plt.show()
    display(table.round(4))


In [ ]:
# ============================================================
# CELL 53 — AUC: functions + cache (NO plots here)
# Purpose: pixel-level PR-AUC / ROC-AUC (one-vs-rest, 7 classes) on the TEST split for every model.
# SegFormer outputs a real per-pixel softmax, so no score-map approximation is needed.
# Pixels are sampled at random, 8,000 per image, on a 160x160 grid; the sample is cached on disk.
# ============================================================
from sklearn.metrics import (precision_recall_curve, roc_curve, average_precision_score, auc as sk_auc)

AUC_SIZE = 160
AUC_PIXELS_PER_IMAGE = 8000
AUC_MAX_PLOT_POINTS = 2000


def collect_pixel_scores(method, split="test"):
    cache = EV / f"{split}_{method}_pixel_scores.npz"
    if cache.exists():
        z = np.load(cache)
        return z["y"], z["s"]

    model = load_best_model(method)
    rng = np.random.default_rng(SEED)
    loader = test_loader if split == "test" else val_loader
    ys, ss = [], []
    with torch.no_grad():
        for x, _, gts, uids in tqdm(loader, desc=f"scores {method} {split}"):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                logits = model(pixel_values=x.to(DEVICE)).logits
            probs = F.interpolate(logits.float(), size=(AUC_SIZE, AUC_SIZE), mode="bilinear",
                                  align_corners=False).softmax(dim=1).cpu().numpy()
            for j in range(len(uids)):
                gt = cv2.resize(gts[j], (AUC_SIZE, AUC_SIZE), interpolation=cv2.INTER_NEAREST)
                idx = rng.choice(AUC_SIZE * AUC_SIZE, size=AUC_PIXELS_PER_IMAGE, replace=False)
                ys.append(gt.ravel()[idx])
                ss.append(probs[j].reshape(NUM_CLASSES, -1)[:, idx].T)
    y, s = np.concatenate(ys), np.concatenate(ss).astype(np.float32)
    np.savez_compressed(cache, y=y, s=s)
    del model
    cleanup_gpu()
    return y, s


def thin(x, y):
    if len(x) <= AUC_MAX_PLOT_POINTS:
        return x, y
    keep = np.linspace(0, len(x) - 1, AUC_MAX_PLOT_POINTS).astype(int)
    return x[keep], y[keep]


def auc_tables(y, s):
    out = {}
    for k in range(NUM_CLASSES):
        t = (y == k).astype(int)
        if t.sum() == 0 or t.sum() == len(t):
            out[k] = None
            continue
        sc = s[:, k]
        p, r, _ = precision_recall_curve(t, sc)
        fpr, tpr, _ = roc_curve(t, sc)
        out[k] = dict(pr=thin(r, p), roc=thin(fpr, tpr),
                      ap=float(average_precision_score(t, sc)), roc_auc=float(sk_auc(fpr, tpr)))
    return out


AUC_RESULTS = globals().get("AUC_RESULTS", {})


def get_auc(method):
    if method not in AUC_RESULTS:
        y, s = collect_pixel_scores(method, "test")
        AUC_RESULTS[method] = auc_tables(y, s)
        del y, s
        gc.collect()
    return AUC_RESULTS[method]


def plot_auc_panels(kind):
    """ONE figure with one panel per model. kind = 'pr' or 'roc'."""
    fig, axes = plt.subplots(1, len(EXPS), figsize=(7.2 * len(EXPS), 7.6))
    for ax, m in zip(axes, EXPS):
        for k, v in get_auc(m).items():
            if v is None:
                continue
            x, yv = v["pr"] if kind == "pr" else v["roc"]
            val = v["ap"] if kind == "pr" else v["roc_auc"]
            ax.plot(x, yv, lw=2.4, color=CLASS_COLORS_7[k],
                    label=f"{MASK_NAMES[k].replace('_', ' ').title()}  ({val:.2f})")
        if kind == "roc":
            ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.5)
        ax.set_xlabel("Recall" if kind == "pr" else "False positive rate")
        ax.set_ylabel("Precision" if kind == "pr" else "True positive rate")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ax.set_title(MODEL_LABELS_1LINE[m], fontsize=14, fontweight="bold")
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=11)
    fig.suptitle("SegFormer - " + ("Precision Recall Curves" if kind == "pr" else "ROC Curves"),
                 fontsize=16, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG / f"auc_{kind}_all_models.png", dpi=130, bbox_inches="tight")
    plt.show()


print("AUC helpers ready. Next: the curves cell, then the AUC summary cell.")


In [ ]:
# CELL 54 — AUC curves: all four SegFormer runs (PR curves, then ROC curves)
plot_auc_panels("pr")
plot_auc_panels("roc")


In [ ]:
# ============================================================
# CELL 55 — AUC SUMMARY TABLE + BARS (writes test_auc_summary.csv)
# Purpose: Macro PR-AUC and ROC-AUC over the 7 classes and over the 6 foreground classes.
# ============================================================
auc_rows, per_class_auc = [], {}
for m in EXPS:
    res = get_auc(m)
    auc_rows.append({
        "model": m,
        "macro_PR_AUC_7": np.mean([v["ap"] for v in res.values() if v is not None]),
        "macro_ROC_AUC_7": np.mean([v["roc_auc"] for v in res.values() if v is not None]),
        "macro_PR_AUC_fg6": np.mean([res[k]["ap"] for k in range(1, NUM_CLASSES) if res[k] is not None]),
        "macro_ROC_AUC_fg6": np.mean([res[k]["roc_auc"] for k in range(1, NUM_CLASSES) if res[k] is not None]),
    })
    per_class_auc[m] = pd.DataFrame({
        "PR_AUC": {MASK_NAMES[k]: (v["ap"] if v else np.nan) for k, v in res.items()},
        "ROC_AUC": {MASK_NAMES[k]: (v["roc_auc"] if v else np.nan) for k, v in res.items()}})

auc_summary = pd.DataFrame(auc_rows).set_index("model")
auc_summary.to_csv(EV / "test_auc_summary.csv")
display(auc_summary.round(4))
pd.concat(per_class_auc, axis=1).to_csv(EV / "test_auc_per_class.csv")
display(pd.concat(per_class_auc, axis=1).round(4))

fig, ax = plt.subplots(figsize=(12, 5.5))
labels = ["PR-AUC (7)", "ROC-AUC (7)", "PR-AUC (fg6)", "ROC-AUC (fg6)"]
width = 0.2
for j, m in enumerate(EXPS):
    for i, v in enumerate(auc_summary.loc[m].values):
        x = i + (j - 1.5) * width
        ax.bar(x, v, width, color=MODEL_COLORS[m], edgecolor="black", label=DISPLAY_NAMES[m] if i == 0 else None)
        ax.text(x, v + 0.01, f"{v:.2f}", ha="center", fontsize=10, rotation=90)
ax.set_xticks(range(4)); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.15); ax.legend(ncol=4, loc="upper right")
ax.set_title("SegFormer - Macro AUC", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "auc_summary_all_models.png")
plt.show()


In [ ]:
# ============================================================
# CELL 56 — FIXED 12-IMAGE PREDICTION OVERLAYS (one figure per model)
# Purpose: Draw the same 12 reference images (reference set cell) with each model's prediction overlaid.
# Saves: <method>/reference12_overlay.png (and a copy in eval/figures)
# ============================================================
for method in EXPS:
    ref_model = load_best_model(method)
    fig, axes = plt.subplots(3, 4, figsize=(20, 16))
    for ax, ref in zip(axes.flat, REFERENCE_12):
        rgb = load_rgb(IMAGE_INDEX["test"][ref["uid"]])
        pred = predict_image(ref_model, rgb)
        gt = load_gt("test", ref["uid"])
        inter = ((pred == gt) & (gt > 0)).sum()
        union = ((pred > 0) | (gt > 0)).sum()
        overlay = np.where((pred > 0)[..., None], (rgb * 0.5 + color_mask(pred) * 0.5).astype(np.uint8), rgb)
        ax.imshow(overlay)
        ax.set_title(f"{ref['class']}\nforeground IoU {inter / max(union, 1):.2f}", fontsize=16)
        ax.axis("off"); ax.grid(False)
    handles = [Patch(color=MASK_COLORS[i] / 255, label=MASK_NAMES[i].title()) for i in range(1, NUM_CLASSES)]
    fig.legend(handles=handles, loc="lower center", ncol=6, fontsize=15, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"{MODEL_LABELS_1LINE[method]} - Test Reference Set Predictions", fontsize=15, fontweight="bold", y=0.99)
    fig.tight_layout(rect=(0, 0.03, 1, 0.97))
    fig.savefig(EXP_ROOTS[method] / "reference12_overlay.png")
    fig.savefig(FIG / f"{method}_reference12_overlay.png")
    plt.show()
    del ref_model
    cleanup_gpu()


In [ ]:
# ============================================================
# CELL 57 — GRAD-CAM GRID ON THE 12 REFERENCE IMAGES (2 figures, 6 images each, all four models)
# Purpose: Same method as the Grad-CAM cell, drawn on the semantic mask of each reference image.
# Columns: semantic mask | baseline | DINOv3 | BYOL | MAE
# ============================================================
cam_cache = {m: [] for m in EXPS}
ref_info = []
for ref in REFERENCE_12:
    gt = load_gt("test", ref["uid"])
    gt512 = cv2.resize(gt, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    ref_info.append((ref, main_class(gt), color_mask(gt512), gt512))

for method in EXPS:
    grid_model = load_best_model(method)
    for ref, class_id, _, gt512 in tqdm(ref_info, desc=f"Grad-CAM {method}"):
        cam, _, _ = gradcam_segformer(grid_model, load_rgb(IMAGE_INDEX["test"][ref["uid"]]), class_id, gt512)
        cam_cache[method].append(cam)
    del grid_model
    cleanup_gpu()

for part, sl in enumerate([slice(0, 6), slice(6, 12)], start=1):
    chunk = list(range(12))[sl]
    fig, axes = plt.subplots(len(chunk), 1 + len(EXPS), figsize=(3.6 * (1 + len(EXPS)), 3.6 * len(chunk)), squeeze=False)
    for r_i, i in enumerate(chunk):
        ref, class_id, mask_rgb, _ = ref_info[i]
        axes[r_i, 0].imshow(mask_rgb)
        axes[r_i, 0].set_title(f"Mask - {MASK_NAMES[class_id]}", fontsize=11)
        for c_i, m in enumerate(EXPS, start=1):
            axes[r_i, c_i].imshow(cam_overlay(mask_rgb, cam_cache[m][i]))
            if r_i == 0:
                axes[r_i, c_i].set_title(DISPLAY_NAMES[m], fontsize=13)
    for ax in axes.flat:
        ax.axis("off"); ax.grid(False)
    sm = plt.cm.ScalarMappable(cmap=_cam_cmap, norm=plt.Normalize(0, 1))
    fig.colorbar(sm, ax=axes, fraction=0.012, pad=0.01, label="Grad-CAM importance (low → high)")
    fig.suptitle(f"SegFormer Grad-CAM ({part} of 2)", fontsize=16, fontweight="bold")
    fig.savefig(FIG / f"gradcam_grid_{part}.png", dpi=110)
    plt.show()
    print("Saved:", FIG / f"gradcam_grid_{part}.png")


In [ ]:
# ============================================================
# CELL 58 — LIME PANELS ON THE 12 REFERENCE IMAGES (2 figures per model, 6 images each)
# Purpose: For every reference image show (1) the semantic mask, (2) the top supporting
#          superpixels, (3) the superpixel weights. Uses use_lime_model() from the LIME cell.
# ============================================================
for method in EXPS:
    use_lime_model(method)
    for part, chunk in enumerate([REFERENCE_12[:6], REFERENCE_12[6:]], start=1):
        fig, axes = plt.subplots(len(chunk), 3, figsize=(16, 3.9 * len(chunk)), squeeze=False)
        for row, ref in enumerate(tqdm(chunk, desc=f"LIME {method} part {part}")):
            _, im = lime_panels(ref["uid"], num_samples=200, axes=axes[row])
        fig.colorbar(im, ax=axes[:, 2], fraction=0.02, pad=0.01, label="LIME weight")
        fig.suptitle(f"LIME - {MODEL_LABELS_1LINE[method]} ({part} of 2)", fontsize=16, fontweight="bold")
        fig.savefig(FIG / f"lime_panels_{method}_{part}.png", dpi=100)
        plt.show()
    cleanup_gpu()


In [ ]:
# ============================================================
# CELL 59 — TEST EMBEDDINGS (embeddings.npy) + PCA / K-MEANS PLOTS (all four runs)
# Purpose: One encoder embedding per test image (global-average-pooled last MiT stage, 512-d),
#          PCA to 2 dimensions, k-means, elbow curve. Test images only pass through best.pt.
# Saves: <method>/embeddings.npy (test images, sorted by uid), eval/embedding_cluster_scores.csv
# ============================================================
label_of = dict(zip(man["uid"], man["label"].astype(int)))
class_names_6 = [n.replace("_", " ").title() for n in CLASS_NAMES]
colors6 = CLASS_COLORS_7[1:]
EMB, cluster_rows = {}, {}

for method in EXPS:
    emb_model = load_best_model(method)
    features, emb_uids = [], []
    with torch.no_grad():
        for x, _, _, uids in tqdm(test_loader, desc=f"embeddings {method}"):
            out = emb_model(pixel_values=x.to(DEVICE), output_hidden_states=True)
            features.append(out.hidden_states[-1].float().mean(dim=(2, 3)).cpu().numpy())
            emb_uids += list(uids)
    EMB[method] = np.concatenate(features)
    np.save(EXP_ROOTS[method] / "embeddings.npy", EMB[method])
    del emb_model
    cleanup_gpu()

    y_test = np.array([label_of[u] - 1 for u in emb_uids])                # 0..5
    Z = PCA(n_components=2, random_state=SEED).fit_transform(EMB[method])
    kmeans_labels = KMeans(n_clusters=NC, n_init=10, random_state=SEED).fit_predict(EMB[method])
    inertia = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(EMB[method]).inertia_ for k in range(2, 11)]
    cluster_rows[method] = {"ARI": adjusted_rand_score(y_test, kmeans_labels),
                            "NMI": normalized_mutual_info_score(y_test, kmeans_labels),
                            "silhouette": float(silhouette_score(EMB[method], y_test))}

    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    for k in range(NC):
        sel = y_test == k
        axes[0].scatter(Z[sel, 0], Z[sel, 1], s=70, color=colors6[k], edgecolor="black", linewidth=0.5,
                        label=class_names_6[k])
    axes[0].set_title(f"{MODEL_LABELS_1LINE[method]} - PCA by true class")
    axes[1].scatter(Z[:, 0], Z[:, 1], s=70, c=kmeans_labels, cmap="tab10", edgecolor="black", linewidth=0.5)
    axes[1].set_title(f"k-means (k={NC})\nARI {cluster_rows[method]['ARI']:.2f} | NMI {cluster_rows[method]['NMI']:.2f}")
    axes[2].plot(range(2, 11), inertia, marker="o", linewidth=2)
    axes[2].set_title("Elbow: within-cluster variance")
    axes[2].set_xlabel("k"); axes[2].set_ylabel("Inertia")
    for ax in axes[:2]:
        ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=6, fontsize=14, markerscale=2, bbox_to_anchor=(0.5, -0.04))
    fig.suptitle(f"{MODEL_LABELS_1LINE[method]} - Embeddings PCA and k-Means", fontsize=15, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG / f"{method}_embeddings_pca_kmeans.png")
    plt.show()

cluster_df = pd.DataFrame(cluster_rows).T
cluster_df.to_csv(EV / "embedding_cluster_scores.csv")
display(cluster_df.round(4))


In [ ]:
# ============================================================
# CELL 60 — COMPLETE COMPARISON MATRIX
# Purpose: One table: 3 methods + baseline x mIoU, Macro-F1, MCC, PR-AUC, accuracy, precision, recall,
#          parameters, GFLOPs, latency/FPS, verdict vs baseline.
# ============================================================
final = pd.read_csv(EV / "test_summary.csv", index_col=0).loc[EXPS][["pixel_accuracy", "mPrecision_fg6", "mRecall_fg6"]]
bt = pd.read_csv(EV / "test_bootstrap_1000.csv", index_col=0).loc[EXPS][
    ["mIoU_7", "mIoU_7_CI_low", "mIoU_7_CI_high", "MacroF1_7", "MacroF1_7_CI_low", "MacroF1_7_CI_high",
     "MCC", "MCC_CI_low", "MCC_CI_high", "mIoU_7_diff", "mIoU_7_diff_low", "mIoU_7_diff_high", "verdict_vs_baseline"]]
auc_t = pd.read_csv(EV / "test_auc_summary.csv", index_col=0).loc[EXPS][["macro_PR_AUC_7", "macro_ROC_AUC_7"]]
comp = pd.read_csv(EV / "compute_profile.csv", index_col=0).loc[EXPS].rename(columns={"estimated_GMACs": "GMACs"})
comp["GFLOPs"] = 2 * comp["GMACs"]                                  # 1 MAC = 2 FLOPs

COMPLETE = bt.join(final).join(auc_t).join(comp)
COMPLETE.to_csv(EV / "final_results_complete.csv")
display(COMPLETE.round(4))

cols = ["mIoU_7", "MacroF1_7", "MCC", "macro_PR_AUC_7", "pixel_accuracy", "mPrecision_fg6", "mRecall_fg6"]
fig, ax = plt.subplots(figsize=(15, 5.5))
data = COMPLETE[cols].values.astype(float)
im = ax.imshow(np.nan_to_num(data), cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(["mIoU", "Macro-F1", "MCC", "PR-AUC", "Accuracy", "Precision", "Recall"], fontsize=15)
ax.set_yticks(range(len(EXPS)))
ax.set_yticklabels([MODEL_LABELS_1LINE[m] for m in EXPS], fontsize=15)
for i in range(len(EXPS)):
    for j in range(len(cols)):
        ax.text(j, i, "n/a" if np.isnan(data[i, j]) else f"{data[i, j]:.3f}", ha="center", va="center",
                fontsize=17, fontweight="bold", color="white" if data[i, j] > 0.6 else "black")
ax.grid(False)
fig.colorbar(im, ax=ax, label="score")
ax.set_title("SegFormer (All Approaches) - Comparison Matrix", fontsize=15, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "comparison_matrix.png")
plt.show()


In [ ]:
# ============================================================
# CELL 61 — VERDICT: does self-supervised pretraining help?
# Purpose: One sentence per method, judged by the paired bootstrap 95% CI of the mIoU difference
#          (SSL minus baseline, same resampled test images): CI above 0 = better, includes 0 = same,
#          below 0 = worse. Baseline start weights are stated below.
# ============================================================
print(f"Baseline start weights: {BASELINE_INIT}. SSL epochs = {SSL_EPOCHS}, fine-tuning epochs = {FT_EPOCHS}.\n")
verdict_rows = []
for m in [x for x in EXPS if x != "baseline"]:
    e = TEST_METRICS[m]
    word = {"BETTER": "better than", "SAME": "the same as", "WORSE": "worse than"}[e["verdict_mIoU_7"]]
    print(f"SegFormer + {DISPLAY_NAMES[m]} is {word} the baseline "
          f"(mIoU difference {e['mIoU_7_diff']:+.4f}, 95% CI [{e['mIoU_7_diff_low']:+.4f}, {e['mIoU_7_diff_high']:+.4f}]).")
    verdict_rows.append({"method": DISPLAY_NAMES[m], "mIoU_diff": e["mIoU_7_diff"], "CI_low": e["mIoU_7_diff_low"],
                         "CI_high": e["mIoU_7_diff_high"], "verdict": e["verdict_mIoU_7"],
                         "MacroF1_verdict": e["verdict_MacroF1_7"], "MCC_verdict": e["verdict_MCC"]})
verdict_df = pd.DataFrame(verdict_rows).set_index("method")
verdict_df.to_csv(EV / "verdict_vs_baseline.csv")
display(verdict_df.round(4))
if SSL_EPOCHS < 20:
    print(f"\nNote: only {SSL_EPOCHS} SSL epochs and {FT_EPOCHS} fine-tuning epochs were run. Treat this as a "
          "pipeline check, not as a final result (the project target is 100 epochs).")


In [ ]:
# ============================================================
# CELL 62 — STEP 8: GRADIO PROTOTYPE (Upload -> Predict mask -> Explain -> Result)
# Purpose: Interactive app using the best-performing SegFormer run (highest test mIoU in the comparison matrix).
# NOT TESTED on Kaggle by the author of this cell: run it once and check.
# ============================================================
import subprocess, sys
# Gradio 5.x only (Gradio 6.x needs huggingface_hub>=1.16, but cell 1 pins <1.0).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gradio>=5,<6", "huggingface_hub>=0.34,<1.0"], check=True)
for _name in [m for m in sys.modules if m.split(".")[0] in ("gradio", "gradio_client")]:
    del sys.modules[_name]
import gradio as gr
print("gradio", gr.__version__)

BEST_METHOD = COMPLETE["mIoU_7"].idxmax()
print("App model:", MODEL_LABELS_1LINE[BEST_METHOD])
APP_MODEL = use_lime_model(BEST_METHOD)                        # LIME and Grad-CAM explain this model


def app_predict(image):
    rgb = np.asarray(Image.fromarray(image).convert("RGB"))
    pred, probs = predict_image(APP_MODEL, rgb, return_probs=True)
    return rgb, pred, color_mask(pred), probs


def app_explain(rgb, pred, probs):
    counts = np.bincount(pred.ravel(), minlength=NUM_CLASSES)[1:]
    if counts.sum() > 0:
        cls = int(np.argmax(counts)) + 1
    else:                                                       # nothing predicted: use top soft class
        cls = int(probs[1:].mean(dim=(1, 2)).argmax().item()) + 1

    mask512 = cv2.resize(pred, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    cam, _, _ = gradcam_segformer(APP_MODEL, rgb, cls, mask512)
    cam_img = cam_overlay(color_mask(mask512), cam)

    lime_rgb = cv2.resize(rgb, LIME_SIZE, interpolation=cv2.INTER_AREA)
    explanation, _ = explain_lime_rgb(lime_rgb, num_samples=150, class_id=cls)
    _, regions = explanation.get_image_and_mask(cls, positive_only=True, num_features=8, hide_rest=False)
    lime_base = color_mask(cv2.resize(pred, LIME_SIZE, interpolation=cv2.INTER_NEAREST)).astype(np.float32) / 255
    lime_img = (mark_boundaries(lime_base, regions, color=(1, .9, 0), mode="thick") * 255).astype(np.uint8)
    return cam_img, lime_img


def app_result(rgb, pred, colored):
    overlay = np.where((pred > 0)[..., None], (rgb * 0.5 + colored * 0.5).astype(np.uint8), rgb)
    total = max((pred > 0).sum(), 1)
    lines = []
    for k in range(1, NUM_CLASSES):
        share = (pred == k).sum() / total * 100
        if share > 0.5:
            lines.append(f"{MASK_NAMES[k].title()}: {share:.1f}% of detected leaf area")
    return overlay, "\n".join(lines) or "No leaf / disease region detected."


def run_pipeline(image):
    if image is None:
        raise gr.Error("Upload a leaf photo first.")
    rgb, pred, colored, probs = app_predict(image)
    cam_img, lime_img = app_explain(rgb, pred, probs)
    overlay, summary = app_result(rgb, pred, colored)
    return colored, cam_img, lime_img, overlay, summary


with gr.Blocks(title="Papaya leaf segmentation") as demo:
    gr.Markdown(f"## Papaya leaf disease segmentation — {MODEL_LABELS_1LINE[BEST_METHOD]}")
    with gr.Row():
        inp = gr.Image(label="1 · Upload leaf photo", type="numpy")
        out_mask = gr.Image(label="2 · Predicted mask")
    with gr.Row():
        out_cam = gr.Image(label="3a · Explain: Grad-CAM")
        out_lime = gr.Image(label="3b · Explain: LIME")
    with gr.Row():
        out_over = gr.Image(label="4 · Result overlay")
        out_txt = gr.Textbox(label="4 · Detected classes", lines=6)
    gr.Button("Run", variant="primary").click(
        run_pipeline, inp, [out_mask, out_cam, out_lime, out_over, out_txt])

demo.launch(share=True, debug=False)


In [ ]:
# ============================================================
# CELL 63 — FINAL RESULTS BACKUP
# Purpose: Package SSL weights, best checkpoints, CSVs, XAI plots and figures into one zip.
# Excluded to keep the zip small: last.pt / *_last.pt (optimizer states used only for resuming) and the
# shared checkpoints/ folder (the SSL weights are already inside each <method>/ssl_weights.pt).
# ============================================================
BACKUP = W / f"papaya_segformer_ssl_e{FT_EPOCHS}_results.zip"
if BACKUP.exists():
    BACKUP.unlink()

with zipfile.ZipFile(BACKUP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob("*")):
        if (path.is_file() and path.name != "last.pt" and not path.name.endswith("_last.pt")
                and CKPT not in path.parents):
            archive.write(path, path.relative_to(OUT.parent))

print("Backup created:", BACKUP)
print("Backup size (MB):", round(BACKUP.stat().st_size / (1024 ** 2), 2))
print("Per run (baseline, dino, byol, mae): ssl_weights.pt, best.pt, history.csv, test_metrics.json, "
      "embeddings.npy, reference12_overlay.png")
print(f"SegFormer DINOv3 / BYOL / MAE / baseline ({FT_EPOCHS}-epoch) pipeline completed.")
